# Bölüm 15: Transformers for Natural Language Processing and Chatbots

> 📚 **Kaynak:** *Hands-On Machine Learning with Scikit-Learn and PyTorch* — Aurélien Géron  
> 🇹🇷 **Açıklamalar:** Türkçe | **Terimler:** İngilizce

---

## 📌 Bölüme Genel Bakış

2017 yılında Google araştırmacıları **"Attention Is All You Need"** adlı çığır açan makalelerinde **Transformer** mimarisini tanıttı. Bu mimari, **recurrent (yinelemeli)** veya **convolutional (evrişimli)** katman içermeyen, tamamen **attention mekanizması** üzerine kurulu bir **encoder-decoder** modelidir.

Bu bölümde şunları öğreneceğiz:
- **Transformer** mimarisini sıfırdan PyTorch ile inşa etmek
- **Multi-Head Attention (MHA)** mekanizması
- **BERT** (encoder-only) ve **GPT** (decoder-only) modellerini anlamak ve kullanmak
- **Sentence-BERT (SBERT)** ile cümle benzerliği hesaplama
- **Prompt engineering** teknikleri
- Sıfırdan **chatbot** inşa etmek
- **SFT (Supervised Fine-Tuning)** ve **DPO (Direct Preference Optimization)** ile modeli hizalama

---

In [1]:
import sys

# sys.version_info: Çalışan Python versiyonunu tuple olarak verir
# (3, 10, 5) gibi. >= (3, 10) karşılaştırması major ve minor versiyonu kontrol eder.
assert sys.version_info >= (3, 10)


In [1]:
conda activate torch_env


CommandNotFoundError: Your shell has not been properly configured to use 'conda activate'.
To initialize your shell, run

    $ conda init <SHELL_NAME>

Currently supported shells are:
  - bash
  - fish
  - tcsh
  - xonsh
  - zsh
  - powershell

See 'conda init --help' for more information and options.

IMPORTANT: You may need to close and restart your shell after running 'conda init'.



Note: you may need to restart the kernel to use updated packages.


In [3]:
# Doğrudan resmi PyTorch kanalından çekiyoruz
!pip install torch==2.2.2 torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu


In [ ]:
# %pip install -qU: pip ile kurulum yap
# -q: quiet (sessiz) modu — uzun çıktıyı gizle
# -U: upgrade — en son sürümü yükle
if IS_COLAB:
    %pip install -qU torchmetrics   # Metrik hesaplama kütüphanesi

if IS_KAGGLE:
    %pip install -qU transformers   # Hugging Face Transformers kütüphanesi


In [ ]:
import torch
from packaging.version import Version

# Kurulu olan PyTorch versiyonunu alalım
current_version = torch.__version__

print(f"Mevcut PyTorch Versiyonu: {current_version}")

# Versiyon kontrolü (En az 2.6.0 olması bekleniyor)
target_version = "2.6.0"

if Version(current_version) >= Version(target_version):
    print(f"✅ Doğrulama başarılı: {current_version} >= {target_version}")
else:
    print(f"❌ HATA: PyTorch versiyonunuz düşük!")
    print(f"Lütfen şu komutla güncelleyin: !pip install --upgrade torch")

# Hata fırlatmasını (Assertion) hala istiyorsanız alt satırı aktif bırakabilirsiniz:
assert Version(current_version) >= Version(target_version), f"Gerekli: {target_version}, Mevcut: {current_version}"

In [4]:
# torch.cuda.is_available(): NVIDIA GPU ve CUDA sürücüsü var mı?
# torch.backends.mps.is_available(): Apple Silicon GPU var mı?
# Önce CUDA'ya bak, sonra MPS'e, ikisi de yoksa CPU kullan
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device   # Seçilen device'ı ekrana yazdır


'mps'

In [6]:
# Eğer GPU bulunamadıysa kullanıcıyı uyar ve nasıl GPU ekleyeceğini göster
if device == "cpu":
    print("Neural nets can be very slow without a hardware accelerator.")
    if IS_COLAB:
        print("Go to Runtime > Change runtime and select a GPU hardware accelerator.")
    if IS_KAGGLE:
        print("Go to Settings > Accelerator and select GPU.")


In [6]:
!pip install "numpy<2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 41.3 kB/s  0:09:53m0:00:0500:22
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Can't uninstall 'numpy'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 1.26.4 which is incompatible.
scipy 1.9.3 requires numpy<1.26.0,>=1.18.5, but you have numpy 1.26.4 which is incompatible.


In [1]:
import matplotlib.pyplot as plt

# plt.rc(): Matplotlib için global stil ayarları
# Tüm grafiklerde bu yazı boyutu ve stil kullanılacak
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)


In [2]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    """
    Modeli evaluation moduna alır ve data_loader üzerindeki metriği hesaplar.
    
    Parametreler:
    - model:       Değerlendirilecek PyTorch modeli
    - data_loader: Değerlendirme verisi
    - metric:      TorchMetrics metrik nesnesi (örn. Accuracy)
    """
    model.eval()       # Dropout ve BatchNorm'u kapat (inference modu)
    metric.reset()     # Önceki epoch'un metrik değerini sıfırla
    with torch.no_grad():  # Gradyan hesaplamasını durdur → bellek ve hız tasarrufu
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()


def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None):
    """
    Tam eğitim döngüsü. ReduceLROnPlateau scheduler ile learning rate yönetimi yapar.
    
    Parametreler:
    - model:          Eğitilecek PyTorch modeli
    - optimizer:      Optimizer (NAdam, Adam, vb.)
    - loss_fn:        Kayıp fonksiyonu (CrossEntropyLoss, vb.)
    - metric:         TorchMetrics metrik nesnesi
    - train_loader:   Eğitim verisi DataLoader'ı
    - valid_loader:   Doğrulama verisi DataLoader'ı
    - n_epochs:       Toplam epoch sayısı
    - patience:       Scheduler'ın kaç epoch iyileşme olmazsa lr'yi düşüreceği
    - factor:         lr'nin kaçla çarpılacağı (0.5 = yarıya indir)
    - epoch_callback: Her epoch başında çağrılan opsiyonel fonksiyon
    """
    # ReduceLROnPlateau: Validation metriği duraksayınca lr'yi otomatik düşür
    # mode='max': Metriği maksimize etmeye çalışıyoruz (accuracy)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}

    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()  # Training moduna al — Dropout aktif

        if epoch_callback is not None:
            epoch_callback(model, epoch)

        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)          # Forward pass
            loss = loss_fn(y_pred, y_batch)  # Kayıp hesapla
            total_loss += loss.item()
            loss.backward()                  # Gradyanları hesapla (backprop)
            optimizer.step()                 # Ağırlıkları güncelle
            optimizer.zero_grad()            # Gradyanları sıfırla (birikmesini önle)
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")

        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)  # Validation metriğine göre lr ayarla
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history


/Users/livanurkaranfil/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
import gc

def del_vars(variable_names=[]):
    """
    Verilen değişken listesini global namespace'ten siler,
    garbage collector'ı çağırır ve CUDA önbelleğini temizler.
    
    Bu fonksiyon büyük modelleri bellekten atmak için kritiktir.
    """
    for name in variable_names:
        try:
            del globals()[name]  # Global değişkeni sil
        except KeyError:
            pass  # Zaten silinmiş değişkenleri sessizce atla

    gc.collect()  # Python'ın çöp toplayıcısını manuel çağır
                  # Referansı kalmayan nesneleri bellekten temizler

    if device == "cuda":
        torch.cuda.empty_cache()  # PyTorch'un CUDA önbelleğini boşalt
                                   # Ayrılmış ama kullanılmayan GPU belleğini serbest bırakır

def free_vram(device):
    """VRAM temizleme için kısa yardımcı fonksiyon."""
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()


In [6]:
!pip install tokenizers numexpr

  Using cached tokenizers-0.22.2-cp39-abi3-macosx_10_12_x86_64.whl.metadata (7.3 kB)
Using cached tokenizers-0.22.2-cp39-abi3-macosx_10_12_x86_64.whl (3.1 MB)


In [7]:
!pip install transformers

  Using cached safetensors-0.7.0-cp38-abi3-macosx_10_12_x86_64.whl.metadata (4.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 49.3 kB/s  0:05:10m0:00:0500:11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.2 kB/s  0:00:12 eta 0:00:01
Using cached safetensors-0.7.0-cp38-abi3-macosx_10_12_x86_64.whl (467 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers] [transformers]ub]


---
## 🔷 BÖLÜM 1: NMT Dataset Hazırlığı

Bu bölüm, önceki bölümdeki (Bölüm 14) İngilizce→İspanyolca çeviri (NMT) veri setini hazırlar. Transformer modelini bu veri üzerinde eğiteceğiz.


### 2.1 Gerekli Import'lar

In [8]:
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import random_split, DataLoader
import tokenizers  # Hugging Face'in hızlı tokenizer kütüphanesi (Rust tabanlı)


In [ ]:
import torch.nn as nn
from torch.utils.data import random_split, DataLoader
import tokenizers  # Hugging Face'in hızlı tokenizer kütüphanesi

# 1. Kütüphaneyi farklı bir isimle içe aktaralım
import datasets as hf_datasets

# 2. Fonksiyonu bu takma isim üzerinden çağıralım
# Örnek: load_dataset yerine hf_datasets.load_dataset
try:
    # 'squad' veya 'glue' gibi bir veri setiyle test edebilirsin
    dataset = hf_datasets.load_dataset("rotten_tomatoes", split="train")
    print("Veri seti başarıyla yüklendi!")
    print(f"Örnek veri: {dataset[0]}")
except AttributeError as e:
    print(f"Hata: {e}")
    print("İpucu: Eğer hala hata alıyorsan, çalışma dizinindeki 'datasets.py' dosyasını silmelisin.")

README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

### 2.2 Dataset İndirme ve Bölme

Tatoeba çevirisinde İngilizce-İspanyolca cümle çiftleri kullanıyoruz.  
Validation setini %80 train / %20 valid olarak böleriz.


In [ ]:
# load_dataset(): Hugging Face Hub'dan dataset indir
# split=[]: Birden fazla split'i aynı anda indir
# 'validation' ve 'test' split'leri ayrı gelir
nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name="eng-spa",
    split=["validation", "test"])

# Validation setini kendi içinde train/valid olarak böl:
# train_size=0.8 → %80'i yeni eğitim seti, %20'si doğrulama seti
# seed=42 → Tekrarlanabilir rastgele bölme
split = nmt_original_valid_set.train_test_split(train_size=0.8, seed=42)
nmt_train_set, nmt_valid_set = split["train"], split["test"]


### 2.3 BPE Tokenizer Eğitimi

**BPE (Byte Pair Encoding):** Sık geçen karakter çiftlerini birleştirerek sub-word tokenlar oluşturur.  
Örneğin "playing" → ["play", "##ing"] şeklinde bölünebilir.  
Bu sayede kelime dağarcığı dışındaki (OOV) kelimeleri bile temsil edebilir.

Özel token'lar:
- `<pad>` (id=0): Padding — kısa cümleleri doldurmak için
- `<unk>` (id=1): Unknown — kelime dağarcığında olmayan tokenlar için
- `<s>` (id=2): Start-of-sequence (başlangıç)
- `</s>` (id=3): End-of-sequence (bitiş)


In [ ]:
def train_eng_spa():
    """
    İngilizce ve İspanyolca metinleri sırayla yield eden generator fonksiyonu.
    
    BPE tokenizer her iki dili de aynı kelime dağarcığıyla öğrenecek.
    Bu sayede paylaşılan sub-word birimleri her iki dil için de geçerli olacak.
    """
    for pair in nmt_train_set:
        yield pair["source_text"]   # İngilizce cümle
        yield pair["target_text"]   # İspanyolca cümle

max_length = 500    # Token dizisinin maksimum uzunluğu
vocab_size = 10_000 # Kelime dağarcığı boyutu

# BPE model oluştur — bilinmeyen tokenlar için <unk> kullanır
nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)

# Padding: Farklı uzunluktaki dizileri aynı boyuta getir
# pad_id=0 → <pad> tokenının ID'si
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")

# Truncation: max_length'i aşan dizileri kes
nmt_tokenizer.enable_truncation(max_length=max_length)

# Pre-tokenizer: Whitespace'e göre kelimelere böl (hızlı ön işlem)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()

# BPE Trainer: Kelime dağarcığını ve birleştirme kurallarını öğrenir
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size,
    special_tokens=["<pad>", "<unk>", "<s>", "</s>"])

# Tokenizer'ı eğit — generator'dan gelen İngilizce ve İspanyolca metinler üzerinde
nmt_tokenizer.train_from_iterator(train_eng_spa(), nmt_tokenizer_trainer)


### 2.4 NmtPair Veri Yapısı

Bir çeviri çiftini (kaynak ve hedef dil) tutmak için `namedtuple` kullanıyoruz.  
`namedtuple`, normal tuple gibi davranır ama alanlara isimle erişim sağlar.  
`.to(device)` metodu, tüm tensor'ları GPU'ya taşır.


In [ ]:
from collections import namedtuple

# NmtPair: 4 alan içeren isimli tuple
# src_token_ids: Kaynak (İngilizce) token ID'leri  → (batch, src_len)
# src_mask:      Kaynak padding maskesi            → (batch, src_len)
# tgt_token_ids: Hedef (İspanyolca) token ID'leri → (batch, tgt_len)
# tgt_mask:      Hedef padding maskesi             → (batch, tgt_len)
fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]

class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        """Tüm tensor'ları istenen device'a (GPU/CPU) taşır."""
        return NmtPair(
            self.src_token_ids.to(device),
            self.src_mask.to(device),
            self.tgt_token_ids.to(device),
            self.tgt_mask.to(device)
        )


### 2.5 Collate Fonksiyonu ve DataLoader'lar

**Collate fonksiyonu:** `DataLoader`'ın her batch oluştururken çağırdığı fonksiyon.  
Ham metin verilerini tokenize eder ve tensor'lara dönüştürür.

**Önemli Detay:** Teacher Forcing  
Eğitimde decoder'a giren dizi (`tgt_token_ids`) hedef cümlenin başından sondan-bir-öncesi kadar olan kısımdır (`[:, :-1]`).  
Label (`labels`) ise bir pozisyon kaydırılmış hali (`[:, 1:]`) — yani "bir sonraki token nedir?" sorusunun cevabı.


In [ ]:
def nmt_collate_fn(batch):
    """
    DataLoader'ın her batch için çağırdığı fonksiyon.
    Ham metin listesini tokenize ederek NmtPair + label tensor çiftine çevirir.
    
    Teacher Forcing mantığı:
    - tgt_token_ids (giriş): <s> w1 w2 w3    (son token hariç)
    - labels (hedef):             w1 w2 w3 </s>  (ilk token hariç)
    Model her pozisyonda bir sonraki tokeni tahmin etmeyi öğrenir.
    """
    src_texts = [pair['source_text'] for pair in batch]
    # Hedef metinlerin başına <s>, sonuna </s> ekle
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]

    # Batch halinde tokenize et (hızlı, padding otomatik)
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)

    # Token ID listeleri → tensor
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])

    # Attention mask: 1 = gerçek token, 0 = padding
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])

    # Giriş: hedef dizinin ilk token'dan sondan-bir-öncesine (teacher forcing)
    inputs = NmtPair(src_token_ids, src_mask,
                     tgt_token_ids[:, :-1],  # <s> w1 w2 w3
                     tgt_mask[:, :-1])

    # Label: hedef dizinin 2. token'dan sonuna (bir pozisyon kaydırılmış)
    labels = tgt_token_ids[:, 1:]             # w1 w2 w3 </s>
    return inputs, labels

batch_size = 64

# shuffle=True: Her epoch'ta training verisini karıştır → better generalization
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn, shuffle=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn)
nmt_test_loader  = DataLoader(nmt_test_set,  batch_size=batch_size,
                              collate_fn=nmt_collate_fn)


---
## 🔷 BÖLÜM 2: Attention Is All You Need — Transformer Mimarisi

2017 yılında Google araştırmacıları tarafından yayınlanan **"Attention Is All You Need"** makalesi, derin öğrenmeye yeni bir dönem açtı. Transformer mimarisi RNN veya CNN içermez; tamamen **attention mekanizması** üzerine kuruludur.

### Neden Transformer?

| Özellik | RNN/LSTM | Transformer |
|---------|----------|-------------|
| Paralel eğitim | ❌ Sıralı | ✅ Tam paralel |
| Uzun mesafe bağımlılık | ❌ Zayıf | ✅ Güçlü |
| Vanishing gradient | ❌ Sorun | ✅ Skip connection ile çözüldü |
| GPU kullanımı | ❌ Verimsiz | ✅ Çok verimli |


### 2.1: Positional Embedding — Konum Bilgisi Ekleme

### Neden Positional Embedding Gerekir?

Transformer mimarisinde **recurrent (LSTM/GRU gibi)** katman yoktur. Bu yüzden model, token'ların sırasını (konumunu) otomatik olarak bilmez. Örneğin "Kedi fareyi yedi" ile "Fare kediyi yedi" cümleleri aynı token'lardan oluşur ama anlam tamamen farklıdır.

Bunu çözmek için her token'ın embedding vektörüne, o token'ın **sıradaki konumunu** temsil eden bir vektör eklenir. Buna **Positional Embedding** denir.

### Öğrenilebilir vs. Sabit Positional Encoding
- **Orijinal Transformer makalesi** sin/cos fonksiyonlarıyla sabit bir encoding kullandı.
- **Modern modellerin çoğu** (GPT-2, BERT dahil) pozisyon vektörlerini **öğrenilebilir parametre** olarak kullanır — bu yaklaşımı aşağıdaki kodda görüyoruz.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionalEmbedding(nn.Module):
    """
    Öğrenilebilir Positional Embedding katmanı.

    Bu sınıf, token embedding vektörlerine konum bilgisi ekler.
    Konum vektörleri sabit değil, eğitim sırasında öğrenilir (nn.Parameter).

    Parametreler:
    - max_length: Desteklenen maksimum sequence (cümle) uzunluğu
    - embed_dim:  Her token'ın embedding boyutu (örn. 128, 256, 512)
    - dropout:    Overfitting'i önlemek için dropout oranı (varsayılan 0.1)
    """
    def __init__(self, max_length, embed_dim, dropout=0.1):
        super().__init__()

        # pos_embed: (max_length, embed_dim) boyutunda öğrenilebilir parametre
        # torch.randn ile küçük rastgele değerlerle başlatılır (*0.02)
        # nn.Parameter olduğu için optimizer tarafından güncellenecek
        self.pos_embed = nn.Parameter(torch.randn(max_length, embed_dim) * 0.02)

        # Dropout: Eğitim sırasında bazı nöronları rastgele sıfırlayarak
        # overfitting'i önler
        self.dropout = nn.Dropout(dropout)

    def forward(self, X):
        """
        X: (batch_size, seq_len, embed_dim) boyutunda token embedding tensörü

        İşlem:
        1. pos_embed'den sadece ilk X.size(1) kadar konum vektörü al
           Yani sequence ne kadar uzunsa, o kadar konum vektörü kullan.
        2. Token embedding'e konum vektörünü EKLE (toplama işlemi)
        3. Dropout uygula
        """
        return self.dropout(X + self.pos_embed[:X.size(1)])

### 2.2 Positional Embedding Test

Bir batch'e positional embedding uygulayarak şeklin korunduğunu doğrulayalım.

In [ ]:
embed_dim = 512
pos_embedding = PositionalEmbedding(max_length, embed_dim)

# Rastgele token embedding'leri: 256 cümle, her biri 500 token, her token 512 boyutlu
embeddings = torch.randn(256, 500, 512)

# Positional embedding ekle
embeddings_with_pos = pos_embedding(embeddings)

# Giriş ve çıkış şekli aynı olmalı — sadece değerler değişti
embeddings_with_pos.shape  # Beklenen: torch.Size([256, 500, 512])


---
### 2.3 Multi-Head Attention (MHA) — Çok Başlıklı Dikkat Mekanizması

#### Attention Mekanizması Nedir?

Bir token'ı işlerken diğer token'lara **ne kadar dikkat edeceğimizi** öğrenen mekanizma.

Örnek: "The animal didn't cross the street because **it** was tired"  
→ "it" kelimesi "animal"a mı yoksa "street"e mi atıfta bulunuyor?  
Attention mekanizması bunu öğrenebilir.

#### Scaled Dot-Product Attention Formülü:
```
Attention(Q, K, V) = softmax(Q × Kᵀ / √d_k) × V
```

- **Q (Query):** "Neyi arıyorum?" — mevcut token'ın sorusu
- **K (Key):** "Ben neyim?" — diğer token'ların etiketleri
- **V (Value):** "Benim içeriğim nedir?" — token'ların taşıdığı bilgi
- **√d_k ile bölme:** Boyut büyüdükçe dot-product patlar, ölçekleme bunu önler

#### Neden Çok Başlık (Multi-Head)?

Tek başlık → tek tip ilişki öğrenir.  
Çok başlık → her başlık **farklı ilişki türü** öğrenir (sözdizimsel, anlamsal, uzun-mesafeli vb.)

#### Boyut Kısaltmaları:
- `B`: batch size (kaç cümle aynı anda işleniyor)
- `h`: head sayısı (kaç paralel attention başlığı var)
- `Lq`: query sequence uzunluğu
- `Lk`: key sequence uzunluğu
- `d`: embed_dim // h (her başlığın boyutu)


In [ ]:
class MultiheadAttention(nn.Module):
    """
    Sıfırdan yazılmış Multi-Head Attention (MHA) katmanı.
    
    PyTorch'un nn.MultiheadAttention modülüyle eşdeğer sonuç verir
    (batch_first=True, need_weights=True ayarlarıyla).
    Bu implementasyon, mekanizmayı anlamak için eğitim amaçlıdır.
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.h = num_heads          # Başlık sayısı (h)
        self.d = embed_dim // num_heads  # Her başlığın boyutu (d = embed_dim / h)
                                         # Örn: embed_dim=512, h=8 → d=64

        # Q, K, V için ayrı Linear projeksiyonlar
        # Her biri: (embed_dim) → (embed_dim) boyutuna dönüştürür
        # Ağırlıklar eğitim sırasında öğrenilir
        self.q_proj = nn.Linear(embed_dim, embed_dim)  # Query projeksiyonu
        self.k_proj = nn.Linear(embed_dim, embed_dim)  # Key projeksiyonu
        self.v_proj = nn.Linear(embed_dim, embed_dim)  # Value projeksiyonu

        # Tüm başlıkların çıktıları birleştirildikten sonra son projeksiyon
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        # Attention weight'lerine dropout — bazı attention bağlantılarını sıfırla
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, X):
        """
        Embedding boyutunu h başlığa böl.
        
        Giriş: (B, L, embed_dim)
        Çıkış: (B, h, L, d)
        
        Adımlar:
        1. view(): (B, L, embed_dim) → (B, L, h, d)
           embed_dim = h × d olduğundan bu yeniden şekillendirme geçerlidir
        2. transpose(1,2): (B, L, h, d) → (B, h, L, d)
           Başlık boyutunu öne alıyoruz ki her başlık bağımsız işlensin
        """
        return X.view(X.size(0), X.size(1), self.h, self.d).transpose(1, 2)

    def forward(self, query, key, value, attn_mask=None, key_padding_mask=None):
        """
        Multi-Head Attention hesaplama.
        
        Giriş:
        - query: (B, Lq, embed_dim) — soru soran token'lar
        - key:   (B, Lk, embed_dim) — etiket token'ları
        - value: (B, Lk, embed_dim) — içerik token'ları (Lv = Lk)
        
        Not: Self-attention'da query=key=value (aynı dizi)
             Cross-attention'da query decoder'dan, key/value encoder'dan gelir
        """

        # ── Adım 1: Linear Projeksiyonlar + Başlıklara Böl ──
        # Her projeksiyon bağımsız öğrenilir, farklı "bakış açıları" oluşturur
        q = self.split_heads(self.q_proj(query))  # (B, h, Lq, d)
        k = self.split_heads(self.k_proj(key))    # (B, h, Lk, d)
        v = self.split_heads(self.v_proj(value))  # (B, h, Lk, d)

        # ── Adım 2: Scaled Dot-Product Attention Skoru ──
        # q @ k.T: Her query ile her key arasındaki benzerlik skoru
        # (B,h,Lq,d) @ (B,h,d,Lk) = (B,h,Lq,Lk) — her Q-K çiftine bir skor
        # / self.d**0.5: √d_k ile ölçekleme — büyük boyutlarda softmax'ın
        # doygunluğa girmesini önler (gradyan akışını korur)
        scores = q @ k.transpose(2, 3) / self.d**0.5  # (B, h, Lq, Lk)

        # ── Adım 3: Maskeleme (gerekirse) ──
        if attn_mask is not None:
            # attn_mask True olan pozisyonları -∞ yap
            # softmax(-∞) = 0 → o tokena hiç dikkat edilmez
            # Decoder'da gelecek token'ları engellemek için kullanılır (causal mask)
            scores = scores.masked_fill(attn_mask, -torch.inf)  # (B, h, Lq, Lk)

        if key_padding_mask is not None:
            # Padding token'larına dikkat edilmemeli
            # key_padding_mask: (B, Lk) → (B, 1, 1, Lk) — broadcast için boyut ekle
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)  # (B, 1, 1, Lk)
            scores = scores.masked_fill(mask, -torch.inf)       # (B, h, Lq, Lk)

        # ── Adım 4: Softmax → Attention Weights ──
        # Her query için tüm key'lere olan dikkat ağırlıkları toplamı = 1
        # dim=-1: Son boyut (Lk) üzerinde softmax uygula
        weights = scores.softmax(dim=-1)  # (B, h, Lq, Lk)

        # ── Adım 5: Ağırlıklı Value Toplamı ──
        # Dropout: Bazı attention bağlantılarını rastgele sıfırla
        # weights @ v: Her query, value'ların ağırlıklı ortalamasını alır
        Z = self.dropout(weights) @ v  # (B, h, Lq, d)

        # ── Adım 6: Başlıkları Birleştir ──
        Z = Z.transpose(1, 2)   # (B, Lq, h, d)  — başlık boyutunu geri al
        Z = Z.reshape(Z.size(0), Z.size(1), self.h * self.d)  # (B, Lq, embed_dim)

        # ── Adım 7: Son Projeksiyon ──
        # Birleştirilmiş başlık çıktılarını son kez dönüştür
        return (self.out_proj(Z), weights)  # (B, Lq, embed_dim), weights


---
### 2.4 TransformerEncoderLayer — Encoder Katmanı

Bir Transformer Encoder katmanı 2 alt katmandan oluşur:

```
Giriş (src)
    ↓
[1] Multi-Head Self-Attention
    ↓
[Add & Norm]  → Residual Connection + Layer Normalization
    ↓
[2] Feed-Forward Network (FFN)
    ↓
[Add & Norm]  → Residual Connection + Layer Normalization
    ↓
Çıkış
```

**Residual (Skip) Connection:** `output = LayerNorm(input + SubLayer(input))`  
Gradyanların derin ağlarda kaybolmasını önler.

**Layer Normalization vs Batch Normalization:**  
Batch Norm, batch boyutu üzerinden normalize eder → değişken uzunluktaki NLP dizilerinde sorunlu.  
Layer Norm, her örneği kendi feature boyutu üzerinden normalize eder → NLP için standarttır.

**Feed-Forward Network (FFN):**  
Her token bağımsız olarak 2 katlı MLP'den geçer: `Linear → ReLU → Linear`  
İç boyut genellikle embed_dim × 4 (örn. 512 → 2048 → 512).


In [ ]:
class TransformerEncoderLayer(nn.Module):
    """
    Tek bir Transformer Encoder katmanı.
    
    İçerik:
    - Multi-Head Self-Attention: Her token diğer tüm token'lara bakabilir
    - Feed-Forward Network (FFN): Token başına bağımsız MLP
    - 2× Residual Connection + Layer Normalization
    
    Parametreler:
    - d_model:         Embedding boyutu (her token'ın vektör boyutu)
    - nhead:           MHA başlık sayısı (d_model'in katı olmalı)
    - dim_feedforward: FFN'nin gizli katman boyutu (varsayılan: d_model × 4)
    - dropout:         Dropout oranı
    """
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()

        # Multi-Head Self-Attention
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)

        # Feed-Forward Network: iki Linear katman
        self.linear1 = nn.Linear(d_model, dim_feedforward)  # Genişletme
        self.linear2 = nn.Linear(dim_feedforward, d_model)  # Daraltma

        self.dropout = nn.Dropout(dropout)

        # İki ayrı Layer Normalization:
        # norm1: Self-Attention çıktısından sonra
        # norm2: Feed-Forward çıktısından sonra
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        """
        src: (B, seq_len, d_model) — encoder'a giren token embedding'leri
        src_mask: Attention mask (genellikle Encoder'da None)
        src_key_padding_mask: Padding mask — PAD tokenlarını gizler

        Adım 1: Self-Attention + Residual + LayerNorm
        Adım 2: Feed-Forward + Residual + LayerNorm
        """
        # ── Adım 1: Multi-Head Self-Attention ──
        # src hem query hem key hem value olarak verilir (self-attention)
        # Yani her token, diğer tüm token'lara dikkat eder
        attn, _ = self.self_attn(src, src, src,
                                 attn_mask=src_mask,
                                 key_padding_mask=src_key_padding_mask)

        # Residual Connection: src (orijinal) + dropout(attention çıktısı)
        # Layer Normalization: mean=0, std=1'e normalize et
        Z = self.norm1(src + self.dropout(attn))

        # ── Adım 2: Position-wise Feed-Forward Network ──
        # Her token bağımsız olarak aynı 2 katmanlı MLP'den geçer
        # linear1 → ReLU aktivasyon → dropout → linear2 → dropout
        ff = self.dropout(
            self.linear2(
                self.dropout(
                    self.linear1(Z).relu()  # ReLU: negatif değerleri sıfırla
                )
            )
        )

        # Residual Connection + Layer Normalization
        return self.norm2(Z + ff)


---
### 2.5 TransformerDecoderLayer — Decoder Katmanı

Decoder, Encoder'dan **farklı olarak 3 alt katman** içerir:

```
Hedef Giriş (tgt) — şimdiye kadar üretilen hedef dizi
    ↓
[1] Masked Self-Attention  ← Sadece geçmiş tokenlara bakabilir (causal mask)
    ↓
[Add & Norm]
    ↓
[2] Cross-Attention (Encoder-Decoder Attention)  ← Encoder çıktısına bakılır
    ↓                 Query: Decoder'dan | Key & Value: Encoder'dan
[Add & Norm]
    ↓
[3] Feed-Forward Network
    ↓
[Add & Norm]
    ↓
Çıkış
```

**Cross-Attention'ın amacı:** Decoder, çeviri üretirken kaynak cümleye bakabilmek için encoder çıktısını (memory) kullanır.


In [ ]:
class TransformerDecoderLayer(nn.Module):
    """
    Tek bir Transformer Decoder katmanı.
    
    Encoder'dan farklı: 3 alt katman içerir
    1. Masked Self-Attention (causal — gelecek görmez)
    2. Cross-Attention (encoder memory'sine dikkat eder)
    3. Feed-Forward Network
    """
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()

        # 1. Masked Self-Attention: Decoder kendi geçmişine bakar
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)

        # 2. Cross-Attention: Decoder, encoder çıktısına (memory) bakar
        self.multihead_attn = MultiheadAttention(d_model, nhead, dropout)

        self.dropout = nn.Dropout(dropout)

        # Feed-Forward Network (Encoder ile aynı yapı)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        # 3 ayrı Layer Normalization (Encoder'da 2 vardı)
        self.norm1 = nn.LayerNorm(d_model)  # Masked Self-Attention sonrası
        self.norm2 = nn.LayerNorm(d_model)  # Cross-Attention sonrası
        self.norm3 = nn.LayerNorm(d_model)  # FFN sonrası

    def forward(self, tgt, memory,
                tgt_mask=None, memory_mask=None,
                tgt_key_padding_mask=None, memory_key_padding_mask=None):
        """
        Parametreler:
        - tgt:    (B, tgt_len, d_model) — decoder giriş token embedding'leri
        - memory: (B, src_len, d_model) — encoder'ın çıktısı (kaynak dil bilgisi)
        - tgt_mask:               Causal mask (gelecek token'ları gizler)
        - memory_mask:            Memory için attention mask (genellikle None)
        - tgt_key_padding_mask:   Hedef padding maskesi
        - memory_key_padding_mask: Kaynak padding maskesi
        """

        # ── Adım 1: Masked Self-Attention ──
        # Decoder kendi token'larına bakar AMA causal mask sayesinde
        # yalnızca kendi geçmişini ve kendini görebilir (gelecek tokenlar gizli)
        # Bu, eğitimde "hile yapmayı" önler
        attn1, _ = self.self_attn(tgt, tgt, tgt,
                                  attn_mask=tgt_mask,
                                  key_padding_mask=tgt_key_padding_mask)
        Z = self.norm1(tgt + self.dropout(attn1))

        # ── Adım 2: Cross-Attention (Encoder-Decoder Attention) ──
        # Query: Decoder'dan gelir (Z) — "kaynak cümlede ne arıyorum?"
        # Key & Value: Encoder çıktısından (memory) — "kaynak cümlede ne var?"
        # Bu sayede decoder, her adımda kaynak cümlenin farklı kısımlarına odaklanır
        attn2, _ = self.multihead_attn(Z, memory, memory,
                                       attn_mask=memory_mask,
                                       key_padding_mask=memory_key_padding_mask)
        Z = self.norm2(Z + self.dropout(attn2))

        # ── Adım 3: Feed-Forward Network ──
        ff = self.dropout(self.linear2(self.dropout(self.linear1(Z).relu())))
        return self.norm3(Z + ff)


---
### 2.6 TransformerEncoder — Çok Katmanlı Encoder

Birden fazla `TransformerEncoderLayer`'ı üst üste yığarak derin bir encoder oluşturur.  
Her katman bir öncekinin çıktısını giriş olarak alır.  
`deepcopy()` ile her katman bağımsız ağırlıklara sahip olur.


In [ ]:
from copy import deepcopy

class TransformerEncoder(nn.Module):
    """
    Çok katmanlı Transformer Encoder.
    
    num_layers adet TransformerEncoderLayer'ı sırayla uygular.
    Opsiyonel son Layer Normalization (norm) desteği vardır.
    """
    def __init__(self, encoder_layer, num_layers, norm=None):
        super().__init__()
        # ModuleList: Katman listesini PyTorch'a kaydet
        # deepcopy: Her katman kendi bağımsız ağırlıklarına sahip olsun
        # (aynı nesneyi kopyalamak yerine bağımsız kopya)
        self.layers = nn.ModuleList([deepcopy(encoder_layer)
                                     for _ in range(num_layers)])
        self.norm = norm  # Opsiyonel son normalization

    def forward(self, src, mask=None, src_key_padding_mask=None):
        """
        src'yi her katmandan sırayla geçir.
        İlk katmanın çıktısı ikinci katmanın girişi olur (stacking).
        """
        Z = src
        for layer in self.layers:
            Z = layer(Z, mask, src_key_padding_mask)

        # İsteğe bağlı: Tüm katmanlar geçildikten sonra son normalization
        if self.norm is not None:
            Z = self.norm(Z)
        return Z


### 2.7 TransformerDecoder — Çok Katmanlı Decoder

In [ ]:
class TransformerDecoder(nn.Module):
    """
    Çok katmanlı Transformer Decoder.
    
    num_layers adet TransformerDecoderLayer'ı sırayla uygular.
    Her katman hem tgt (hedef) hem de memory (encoder çıktısı) alır.
    """
    def __init__(self, decoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = nn.ModuleList([deepcopy(decoder_layer)
                                     for _ in range(num_layers)])
        self.norm = norm

    def forward(self, tgt, memory,
                tgt_mask=None, memory_mask=None,
                tgt_key_padding_mask=None, memory_key_padding_mask=None):
        """
        tgt'yi her decoder katmanından geçir.
        memory (encoder çıktısı) her katmana iletilir — cross-attention için.
        """
        Z = tgt
        for layer in self.layers:
            Z = layer(Z, memory, tgt_mask, memory_mask,
                      tgt_key_padding_mask, memory_key_padding_mask)
        if self.norm is not None:
            Z = self.norm(Z)
        return Z


### 2.8 Transformer — Tam Encoder-Decoder Modeli

Encoder ve Decoder'ı bir araya getiren üst seviye sınıf.

In [ ]:
class Transformer(nn.Module):
    """
    Tam Transformer modeli (orijinal makale mimarisi).
    
    Encoder + Decoder'ı tek bir sınıfta birleştirir.
    PyTorch'un nn.Transformer modülüyle aynı arayüze sahiptir.
    
    Varsayılan parametreler orijinal makaleye göre:
    - d_model=512, nhead=8, num_layers=6, dim_feedforward=2048
    """
    def __init__(self, d_model=512, nhead=8, num_encoder_layers=6,
                 num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()

        # Encoder oluştur: bir katman template → deepcopy ile çoğalt
        encoder_layer = TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
        norm1 = nn.LayerNorm(d_model)  # Encoder'ın son normalization katmanı
        self.encoder = TransformerEncoder(encoder_layer, num_encoder_layers, norm1)

        # Decoder oluştur
        decoder_layer = TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
        norm2 = nn.LayerNorm(d_model)  # Decoder'ın son normalization katmanı
        self.decoder = TransformerDecoder(decoder_layer, num_decoder_layers, norm2)

    def forward(self, src, tgt,
                src_mask=None, tgt_mask=None, memory_mask=None,
                src_key_padding_mask=None, tgt_key_padding_mask=None,
                memory_key_padding_mask=None):
        """
        src: (B, src_len, d_model) — kaynak dil token embedding'leri
        tgt: (B, tgt_len, d_model) — hedef dil token embedding'leri
        
        1. src → Encoder → memory (bağlamsal kaynak temsili)
        2. tgt + memory → Decoder → output (hedef dil tahminleri)
        """
        # Encoder: Kaynak cümleyi bağlamsal embedding'lere dönüştür
        memory = self.encoder(src, src_mask, src_key_padding_mask)

        # Decoder: Hedef cümleyi oluştururken memory'e bak
        output = self.decoder(tgt, memory, tgt_mask, memory_mask,
                              tgt_key_padding_mask, memory_key_padding_mask)
        return output


---
## 🔷 BÖLÜM 3: İngilizce-İspanyolca Transformer (NmtTransformer)

Şimdiye kadar oluşturduğumuz tüm bileşenleri (PositionalEmbedding, Transformer) birleştirerek tam bir çeviri modeli oluşturuyoruz.

### Causal Mask Nedir?

Decoder eğitimde hedef cümlenin tamamını görür ama her pozisyon yalnızca kendinden **önceki** pozisyonlara bakabilmelidir. Aksi hâlde model "hile yaparak" cevabı görür ve gerçek çeviriyi öğrenmez.

Bu maskeleme için üst üçgen matris kullanılır:
```
      p0  p1  p2  p3
p0: [  F   T   T   T ]   ← p0 sadece kendini görebilir
p1: [  F   F   T   T ]   ← p1 sadece p0 ve p1'i görebilir
p2: [  F   F   F   T ]
p3: [  F   F   F   F ]
```
T (True) = engellenen pozisyon, F (False) = izin verilen pozisyon


In [ ]:
class NmtTransformer(nn.Module):
    """
    Neural Machine Translation için tam Transformer modeli.
    
    Mimari:
    token_ids → Embedding → PositionalEmbedding → Transformer → Linear → logits
    
    Parametreler:
    - vocab_size:  Kelime dağarcığı büyüklüğü (kaynak+hedef için ortak)
    - max_length:  Desteklenen maksimum cümle uzunluğu
    - embed_dim:   Token embedding boyutu (orijinal: 512)
    - pad_id:      <pad> token'ının ID'si (genellikle 0)
    - num_heads:   MHA başlık sayısı (orijinal: 8)
    - num_layers:  Encoder ve Decoder katman sayısı (orijinal: 6)
    - dropout:     Dropout oranı
    """
    def __init__(self, vocab_size, max_length, embed_dim=512, pad_id=0,
                 num_heads=8, num_layers=6, dropout=0.1):
        super().__init__()

        # Embedding: token ID'lerini vektörlere dönüştürür
        # padding_idx=pad_id: PAD token'ının embedding'i her zaman sıfır kalır
        # (PAD token'ı gradyan almaz → eğitimi etkilemez)
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)

        # Öğrenilebilir positional embedding
        self.pos_embed = PositionalEmbedding(max_length, embed_dim, dropout)

        # PyTorch'un built-in Transformer (batch_first=True: modern standart)
        # num_encoder_layers = num_decoder_layers = num_layers
        self.transformer = nn.Transformer(
            embed_dim, num_heads,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            batch_first=True)   # Tensor boyutları: (batch, seq, feature)

        # Son projeksiyon: decoder çıktısını vocab boyutuna dönüştür
        # (B, tgt_len, embed_dim) → (B, tgt_len, vocab_size)
        self.output = nn.Linear(embed_dim, vocab_size)

    def forward(self, pair):
        """
        pair: NmtPair nesnesi — (src_token_ids, src_mask, tgt_token_ids, tgt_mask)
        
        Çıktı: (B, vocab_size, tgt_len) — CrossEntropyLoss için bu format gerekli
        """
        # ── Adım 1: Token ID'lerini embedding'e çevir ──
        src_embeds = self.pos_embed(self.embed(pair.src_token_ids))
        tgt_embeds = self.pos_embed(self.embed(pair.tgt_token_ids))

        # ── Adım 2: Padding maskeleri oluştur ──
        # pair.src_mask: 1=gerçek token, 0=padding
        # nn.Transformer True=engel mantığını kullanır → tersini alıyoruz (~)
        src_pad_mask = ~pair.src_mask.bool()  # (B, src_len) — True = PAD konumu
        tgt_pad_mask = ~pair.tgt_mask.bool()  # (B, tgt_len) — True = PAD konumu

        # ── Adım 3: Causal mask oluştur ──
        # tgt_len × tgt_len boyutunda üst üçgen boolean matrisi
        # torch.triu(M, diagonal=1): Ana diagonalin üstündeki elemanlar True
        size = [pair.tgt_token_ids.size(1)] * 2  # [tgt_len, tgt_len]
        full_mask = torch.full(size, True, device=tgt_pad_mask.device)
        causal_mask = torch.triu(full_mask, diagonal=1)
        # Örnek 4×4 için:
        # [[F, T, T, T],   ← p0 sadece kendini görebilir
        #  [F, F, T, T],   ← p1 sadece p0 ve p1'i
        #  [F, F, F, T],
        #  [F, F, F, F]]

        # ── Adım 4: Transformer'dan geçir ──
        out_decoder = self.transformer(
            src_embeds, tgt_embeds,
            src_key_padding_mask=src_pad_mask,      # Encoder: kaynak PAD'leri gizle
            memory_key_padding_mask=src_pad_mask,   # Cross-att: kaynak PAD'leri gizle
            tgt_mask=causal_mask,                   # Decoder: gelecek token gizle
            # tgt_is_causal=True,                   # PyTorch optimizasyon (yorum satırı)
            tgt_key_padding_mask=tgt_pad_mask       # Decoder: hedef PAD'leri gizle
        )

        # ── Adım 5: Vocab boyutuna projeksiyon + boyutları sırala ──
        # CrossEntropyLoss (B, C, L) formatı bekler:
        # (B, tgt_len, vocab_size) → .permute(0,2,1) → (B, vocab_size, tgt_len)
        return self.output(out_decoder).permute(0, 2, 1)


### 3.1 Causal Mask'i Görselleştirme

Causal mask'in nasıl göründüğünü elle oluşturup görelim.

In [ ]:
# torch.triu(): Üst üçgen matris — diagonal=1 ile ana diagonal hariç
# 5×5 örnek: 5 tokenlik cümle için causal mask
causal_mask_example = torch.triu(torch.full((5, 5), True), diagonal=1)
print("5×5 Causal Mask:")
print(causal_mask_example)
# True = engellenmiş, False = izin verilmiş
# Token 0 sadece kendini, Token 4 hepsini görebilir


### 3.2 PyTorch'un Built-in Causal Mask Fonksiyonu

PyTorch, float maske üreten kendi fonksiyonuna sahiptir.

In [ ]:
# nn.Transformer.generate_square_subsequent_mask():
# 0.0 (izin) ve -inf (engel) değerleri içeren float maske döndürür
# Bu format, bazı attention implementasyonlarında boolean yerine kullanılır
float_mask = nn.Transformer.generate_square_subsequent_mask(5)
print("PyTorch Float Causal Mask:")
print(float_mask)
# 0.0 → izin verilen pozisyon
# -inf → engellenen pozisyon (softmax → 0 ağırlık)


### 3.3 Modeli Oluşturma ve Eğitme

Küçük bir model oluşturuyoruz (orijinal 512/8/6 yerine 128/4/2) — hız için.

**NAdam Optimizer:** Adam optimizer'ın Nesterov momentum iyileştirmesiyle güçlendirilmiş versiyonu.  
**CrossEntropyLoss(ignore_index=0):** PAD token'larının (`id=0`) kaybı hesaba katılmaz — bu çok önemli, aksi hâlde model PAD'leri tahmin etmeye çalışır.

> **MPS Workaround:** Apple Silicon'da `nn.Transformer` eğitim sırasında patlıyor (PyTorch issue #141287). Bu yüzden MPS cihazında kendi yazdığımız Transformer sınıfına geçiyoruz.


In [ ]:
torch.manual_seed(42)

# Küçük model: embed_dim=128, num_heads=4, num_layers=2
# (Orijinal makale: 512, 8, 6 — çok daha büyük ve yavaş)
nmt_tr_model = NmtTransformer(
    vocab_size, max_length,
    embed_dim=128,  # Küçük embedding boyutu → hızlı eğitim
    pad_id=0,
    num_heads=4,    # 4 paralel attention başlığı
    num_layers=2,   # 2 encoder + 2 decoder katmanı
    dropout=0.1
).to(device)  # Modeli GPU'ya taşı

# Apple Silicon (MPS) için workaround
if device == "mps":
    # nn.Transformer MPS'de stabil değil → kendi Transformer sınıfımıza geç
    nmt_tr_model.transformer = Transformer(
        embed_dim=128, num_heads=4, num_encoder_layers=2, num_decoder_layers=2)

n_epochs = 20

# CrossEntropyLoss: Çok sınıflı sınıflandırma kaybı
# ignore_index=0: <pad> token'larını kayıp hesabından çıkar
xentropy = nn.CrossEntropyLoss(ignore_index=0)

# NAdam: Adam + Nesterov momentum — genellikle Adam'dan daha iyi yakınsama
optimizer = torch.optim.NAdam(nmt_tr_model.parameters())

# Doğruluk metriği: Token bazında doğruluk oranı
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size)
accuracy = accuracy.to(device)

# Eğitimi başlat (uzun sürebilir — GPU önerilir)
history = train(nmt_tr_model, optimizer, xentropy, accuracy,
                nmt_train_loader, nmt_valid_loader, n_epochs)


### 3.4 Modeli Kaydetme

In [ ]:
# Modelin sadece ağırlıklarını kaydet (state_dict)
# Bu, modelin mimarisi değil yalnızca öğrenilmiş parametreleri kaydeder
# Yüklemek için: model.load_state_dict(torch.load('my_nmt_tr_model.pt'))
torch.save(nmt_tr_model.state_dict(), "my_nmt_tr_model.pt")


### 3.5 Çeviri Fonksiyonu

Eğitilmiş modeli kullanarak İngilizce metni İspanyolcaya çevirir.

**Autoregressive (Otomatik Gerilimli) Üretim:**  
1. Boş hedef metinle başla
2. Model bir sonraki tokeni tahmin eder
3. Tahmin edilen tokeni hedef metne ekle
4. Tekrar 2'ye git
5. `</s>` (end-of-sequence) görülünce dur


In [ ]:
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    """
    Eğitilmiş NmtTransformer ile İngilizce → İspanyolca çeviri yapar.
    
    Autoregressive decoding: Her adımda bir token üretir.
    
    Parametreler:
    - model:      Eğitilmiş NmtTransformer
    - src_text:   Çevrilecek İngilizce metin
    - max_length: Maksimum çeviri uzunluğu (sonsuz döngüyü önler)
    - pad_id:     Padding token ID'si (0)
    - eos_id:     End-of-sequence token ID'si (3 = </s>)
    """
    tgt_text = ""    # Boş hedef metinle başla
    token_ids = []   # Üretilen token ID'lerini biriktir

    for index in range(max_length):
        # Mevcut (src, tgt) çiftini batch formatına dönüştür
        batch, _ = nmt_collate_fn([{"source_text": src_text,
                                    "target_text": tgt_text}])
        with torch.no_grad():  # Inference: gradyan hesaplama
            Y_logits = model(batch.to(device))        # (1, vocab_size, tgt_len+1)
            Y_token_ids = Y_logits.argmax(dim=1)      # En yüksek olasılıklı token ID
            next_token_id = Y_token_ids[0, index]     # Bu adımdaki tahmin

        # Token ID'sini metne çevir
        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token   # Hedef metne ekle

        # EOS token görülünce çeviri tamamdır
        if next_token_id == eos_id:
            break

    return tgt_text


In [ ]:
# Modeli evaluation moduna al ve test et
nmt_tr_model.eval()
result = translate(nmt_tr_model, "I like to play soccer with my friends at the beach")
print(result)
# Beklenen: ' Me gusta jugar al fútbol con mis amigos en la playa . </s>'


### 3.6 GPU RAM Temizleme

In [ ]:
# NMT modeli ve ilgili tüm nesneleri bellekten temizle
# Sonraki bölümde BERT için yer açıyoruz
del accuracy, history, nmt_test_set, nmt_tokenizer
del nmt_tokenizer_model, nmt_test_loader, nmt_train_loader, nmt_valid_loader
del nmt_tr_model, nmt_train_set, nmt_valid_set, optimizer
del pos_embedding, xentropy
free_vram(device)


---
## 🔷 BÖLÜM 4: Encoder-Only Transformers — BERT ve MLM

### BERT Nedir?

**BERT (Bidirectional Encoder Representations from Transformers)**, 2018'de Google tarafından yayınlandı. Yalnızca Transformer'ın **encoder** kısmını kullanır ve **çift yönlü** (bidirectional) bağlam öğrenir.

- GPT gibi "soldan sağa" değil, hem sol hem sağ komşulara bakabilir
- Pre-training: **Masked Language Modeling (MLM)** ve **Next Sentence Prediction (NSP)**
- Fine-tuning: Metin sınıflandırma, NER, Q&A, vb. görevler için uyarlanır

### BERT'in MLM Pre-training'i

Token'ların %15'i rastgele seçilir:
- **%80'i:** `[MASK]` ile değiştirilir — model maskeli tokeni tahmin eder
- **%10'u:** Rastgele başka bir token ile değiştirilir — modeli sağlamlaştırır
- **%10'u:** Değiştirilmez — model gerçek tokeni de yeniden "tahmin" eder


In [ ]:
from transformers import BertConfig, BertForMaskedLM, BertTokenizerFast

# ── Tokenizer ──
# 'bert-base-uncased': Büyük/küçük harf ayrımı yapmayan, ~30K vocabulary BERT
# WordPiece tokenizer kullanır (BPE'ye benzer)
bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# ── Model Konfigürasyonu ──
# Gerçek BERT-base: hidden_size=768, num_hidden_layers=12, num_attention_heads=12
# Burada KÜÇÜK model: hız ve demo için
config = BertConfig(
    vocab_size=bert_tokenizer.vocab_size,   # ~30,522 token
    hidden_size=128,           # Embedding boyutu (gerçek: 768)
    num_hidden_layers=2,       # Transformer katman sayısı (gerçek: 12)
    num_attention_heads=4,     # MHA başlık sayısı (gerçek: 12)
    intermediate_size=512,     # FFN gizli boyutu (gerçek: 3072 = 768×4)
    max_position_embeddings=128  # Maksimum sequence uzunluğu (gerçek: 512)
)

# ── Model ──
# BertForMaskedLM: BERT encoder + MLM başlığı (Linear + Softmax)
# MLM başlığı: (hidden_size) → (vocab_size) — maskelenmiş tokeni tahmin eder
bert = BertForMaskedLM(config)
# NOT: Ağırlıklar rastgele başlatılır — henüz eğitilmemiş


### 4.1 WikiText Dataset Hazırlığı

MLM pre-training için Wikipedia metni kullanıyoruz.

In [ ]:
from datasets import load_dataset

def tokenize(example, tokenizer=bert_tokenizer):
    """
    Hugging Face dataset'indeki her örneği tokenize eden fonksiyon.
    
    Dataset.map() ile tüm veri setine uygulanacak.
    
    Parametreler:
    - example:   Dataset'ten gelen sözlük ('text' anahtarı içerir)
    - tokenizer: Kullanılacak BERT tokenizer
    
    Tokenizer parametreleri:
    - truncation=True:       128 tokendan uzun metinleri kes
    - max_length=128:        BertConfig'deki max_position_embeddings ile aynı
    - padding='max_length':  Kısa metinleri [PAD] ile 128'e tamamla
                             Batch içindeki tüm örnekler aynı uzunlukta olur
    """
    return tokenizer(example["text"],
                     truncation=True,
                     max_length=128,
                     padding="max_length")

# WikiText-2: Wikipedia'dan derlenmiş ~2M token içeren temiz metin
# MLM pre-training için popüler benchmark veri seti
mlm_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# Tüm dataset'e tokenize fonksiyonunu uygula
# batched=True: Örnekleri toplu işle → çok daha hızlı
mlm_dataset = mlm_dataset.map(tokenize, batched=True)


### 4.2 BERT'i Eğitmek — Hugging Face Trainer API

`Trainer` API, tüm eğitim döngüsünü (forward pass, loss, backward, optimizer, logging, checkpointing) otomatikleştirir.

`DataCollatorForLanguageModeling`: Her batch oluşturulurken token'ları otomatik maskeler.
- `mlm=True`: MLM modu aktif
- `mlm_probability=0.15`: Her token %15 olasılıkla maskelenir


In [ ]:
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling

# Eğitim argümanları
args = TrainingArguments(
    output_dir="./my_bert",           # Checkpoint'ların kaydedileceği klasör
    num_train_epochs=5,               # 5 epoch
    per_device_train_batch_size=16,   # Her GPU'da 16 örnek
    report_to="none"                  # WandB veya TensorBoard'a raporlama yapma
)

# MLM Data Collator: Her batch için dinamik maskeleme uygular
# Bu sayede her epoch'ta farklı token'lar maskelenir → daha iyi genelleme
mlm_collator = DataCollatorForLanguageModeling(
    bert_tokenizer,
    mlm=True,              # Masked Language Modeling aktif
    mlm_probability=0.15   # %15 maskeleme (BERT makalesinden)
)

# Trainer: Yüksek seviyeli eğitim API'si
trainer = Trainer(
    model=bert,
    args=args,
    train_dataset=mlm_dataset,
    data_collator=mlm_collator   # Her batch için maskeleme yapan collator
)

# Eğitimi başlat
# (Küçük model ve az veri bile olsa birkaç dakika sürebilir)
trainer_output = trainer.train()


### 4.3 Fill-Mask Pipeline ile Test

Eğitim sonrası modeli `fill-mask` pipeline'ı ile test ederiz.  
`[MASK]` token'ının yerine ne geleceğini tahmin ederiz.


In [ ]:
from transformers import pipeline

torch.manual_seed(42)

# fill-mask pipeline: [MASK] token'ını tahmin eden araç
fill_mask = pipeline("fill-mask", model=bert, tokenizer=bert_tokenizer)

# Test: "The capital of [MASK] is Rome." → [MASK] = "Italy" beklenir
top_predictions = fill_mask("The capital of [MASK] is Rome.")

# En yüksek olasılıklı tahmin
print(top_predictions[0])
# Muhtemel çıktı: virgül (',') gibi anlamsız bir şey
# ÇÜNKÜ:
# 1. Model çok küçük (128 dim, 2 katman — gerçek BERT 768 dim, 12 katman)
# 2. Çok az veri ile eğitildi (WikiText-2, gerçek BERT C4+Wikipedia)
# 3. Az epoch (5, gerçek BERT yüz binlerce adım)
# Gerçek boyutlu BERT-base ile "Italy" doğru tahmin edilir


In [ ]:
# GPU RAM temizle
del_vars(["bert_tokenizer", "config", "bert", "mlm_dataset", "args",
          "mlm_collator", "trainer", "trainer_output", "fill_mask",
          "top_predictions"])


### 4.4 BERT ile CLS Token Embedding

Pretrained BERT'i doğrudan metin embedding'i için kullanabiliriz.  
`[CLS]` tokeni (0. indeks), BERT'in tüm cümleyi özetlediği özel tokendır.  
Bu embedding, downstream görevler için girdi olarak kullanılabilir.


In [ ]:
from transformers import AutoTokenizer, AutoModel

# Pretrained BERT-base-uncased'i indir (Google'ın eğittiği)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# İki cümleyi tokenize et
# padding=True: Farklı uzunlukları eşitle
# return_tensors='pt': PyTorch tensor olarak döndür
inputs = tokenizer(
    ["I like soccer", "Hello, World!"],
    padding=True,
    return_tensors="pt"
)

# Forward pass: BERT'ten hidden state'leri al
outputs = model(**inputs)

# CLS Token Embedding: outputs.last_hidden_state[:, 0, :]
# last_hidden_state: (batch, seq_len, hidden_size) — son katmanın tüm token çıktıları
# [:, 0, :]: Her cümlenin [CLS] tokeni (0. pozisyon) → cümle temsili
cls_embedding = outputs.last_hidden_state[:, 0, :]

print(f"CLS embedding boyutu: {cls_embedding.shape}")
# Beklenen: (2, 768) — 2 cümle, 768 boyutlu embedding


In [ ]:
# Belleği temizle
del tokenizer, model, inputs, outputs, cls_embedding
free_vram(device)


---
## 🔷 BÖLÜM 5: Sentence-BERT (SBERT) — Cümle Benzerliği

### Neden SBERT?

Standart BERT ile cümle benzerliği hesaplamak için her cümle çiftini birlikte encode etmek gerekir → **O(n²) karmaşıklık**, büyük veri kümelerinde uygun değil.

**SBERT (Sentence-BERT):** Siamese ağ yapısıyla her cümleyi **bağımsız** bir sabit boyutlu vektöre (embedding) dönüştürür.  
Cosine similarity ile hızla karşılaştırılabilir → **O(n) encode, O(n²) karşılaştırma**.

**Kullanım alanları:**
- Anlamsal metin arama (Semantic Search)
- Duplicate tespiti
- Clustering
- RAG (Retrieval-Augmented Generation)


In [ ]:
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2: Popüler, hızlı, hafif SBERT modeli
# 'MiniLM' = knowledge distillation ile küçültülmüş model
# 'L6' = 6 transformer katmanı
# 'v2' = ikinci, iyileştirilmiş versiyon
# 384 boyutlu cümle embedding'leri üretir
model = SentenceTransformer("all-MiniLM-L6-v2")

# Test cümleleri:
# - Cümle 1 ve 2: Her ikisi de alışverişle ilgili → yüksek benzerlik beklenir
# - Cümle 1 ve 3: Farklı konular → düşük benzerlik beklenir
sentences = ["She's shopping", "She bought some shoes", "She's working"]

# Her cümleyi vektöre dönüştür
# convert_to_tensor=True: NumPy yerine PyTorch tensörü döndür
embeddings = model.encode(sentences, convert_to_tensor=True)
# Çıktı boyutu: (3, 384) — 3 cümle, her biri 384 boyutlu vektör

# Tüm çift kombinasyonlarına cosine similarity hesapla
# similarity(A, B): Her (i,j) çifti için A[i] ve B[j] arasındaki cosine benzerliği
similarities = model.similarity(embeddings, embeddings)


In [ ]:
# Benzerlik matrisini görüntüle
print(similarities)
# Beklenen:
# tensor([[1.0000, 0.6328, 0.5841],
#          [0.6328, 1.0000, 0.3831],
#          [0.5841, 0.3831, 1.0000]])
#
# Köşegen: Her cümle kendisiyle 1.0 (mükemmel benzerlik)
# [0,1] = [1,0] = 0.63: "shopping" - "bought shoes" (yüksek benzerlik ✓)
# [0,2] = [2,0] = 0.58: "shopping" - "working" (orta benzerlik)
# [1,2] = [2,1] = 0.38: "bought shoes" - "working" (düşük benzerlik ✓)


In [ ]:
del_vars(["model", "sentences", "embeddings", "similarities"])

---
## 🔷 BÖLÜM 6: Decoder-Only Transformers — GPT-2

### GPT Ailesi: Decoder-Only Transformer

**GPT (Generative Pre-Training)**, 2018'de OpenAI tarafından yayınlandı. BERT'in aksine yalnızca Transformer'ın **decoder** kısmını kullanır.

**Causal Language Modeling (CLM):** Her adımda bir sonraki tokeni tahmin eder:
`P(w_t | w_1, w_2, ..., w_{t-1})`

Bu yüzden GPT metni **soldan sağa** üretir — her token, yalnızca önceki tokenlara bakabilir.

### GPT vs BERT:

| Özellik | BERT | GPT |
|---------|------|-----|
| Mimari | Encoder-only | Decoder-only |
| Yön | Çift yönlü | Tek yönlü (L→R) |
| Pre-training | MLM | CLM |
| Güçlü olduğu görev | Anlama (NLU) | Üretim (NLG) |


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# 'Auto' sınıfları: model_id'ye göre otomatik doğru sınıfı seçer
# Bu sayede aynı kod GPT-2, Mistral, Llama vb. için çalışır
model_id = "gpt2"  # OpenAI'ın açık kaynak GPT-2 modeli (117M parametre)

# Tokenizer: GPT-2 Byte Pair Encoding (BPE) kullanır, ~50K vocabulary
gpt2_tokenizer = AutoTokenizer.from_pretrained(model_id)

# Model yükleme:
# device_map='auto': Mevcut GPU/CPU'ya otomatik yerleştir
#                    Çok büyük modeller için birden fazla GPU'ya dağıtır
# dtype='auto':      Modelin kaydedildiği veri tipini kullan
#                    float16 → yarı bellek kullanımı, hız artışı
gpt2 = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", dtype="auto")


### 6.1 Metin Üretim Fonksiyonu

Bu yardımcı fonksiyon, verilen prompt'tan metin üretir ve tüm bölüm boyunca kullanılır.


In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=50, **generate_kwargs):
    """
    Verilen prompt'tan yeni metin üretir.
    
    Parametreler:
    - model:            Causal LM modeli (GPT-2, Mistral, vb.)
    - tokenizer:        Modele ait tokenizer
    - prompt:           Üretimin başlangıç metni
    - max_new_tokens:   Kaç YENİ token üretileceği (prompt dahil değil)
    - **generate_kwargs: Ek üretim parametreleri (do_sample, top_p, temperature, vb.)
    
    Adımlar:
    1. Prompt'u tokenize et → PyTorch tensörü → model'in device'ına taşı
    2. model.generate() ile token dizisi üret
    3. Token ID'lerini okunabilir metne çevir
    """
    # Tokenize ve device'a taşı
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Token üret
    # pad_token_id=eos_token_id: GPT-2'nin ayrı PAD token'ı yok,
    # EOS (end-of-sequence) token'ını padding olarak kullan
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        **generate_kwargs   # Sampling stratejisi parametreleri buraya geçilir
    )

    # outputs[0]: Batch içindeki ilk (ve tek) üretilen sequence
    # skip_special_tokens=True: [BOS], [EOS] gibi özel token'ları gizle
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


### 6.2 Greedy Decoding — Varsayılan Üretim

**Greedy Decoding:** Her adımda en yüksek olasılıklı tokeni seç.

- ✅ Deterministik: Aynı prompt → her zaman aynı çıktı
- ❌ Tekrar döngüleri: Model aynı cümleyi tekrar edebilir
- ❌ Yaratıcılık yok: Hep "en olası" tercih yapılır


In [ ]:
# do_sample belirtilmezse varsayılan greedy decoding kullanılır
prompt = "Scientists found a talking unicorn today. Here's the full story:"
output = generate(gpt2, gpt2_tokenizer, prompt)
print(output)
# Beklenen sorun: "The unicorn was found in a field..." cümlesi tekrar eder
# Greedy decoding tekrarcı döngülere (repetition loop) düşebilir


### 6.3 Random Sampling — Yaratıcı Üretim

**`do_sample=True`:** Olasılık dağılımından rastgele örnekleme yapar.

- ✅ Çeşitli, yaratıcı metinler
- ✅ Tekrar döngüsü sorunu azalır
- ❌ Bazen tutarsız veya mantıksız sonuçlar
- `torch.manual_seed()` ile tekrarlanabilir sonuç alınır


In [ ]:
torch.manual_seed(42)   # Tekrarlanabilirlik için rastgele sayı üretici sabitle
output_sample = generate(gpt2, gpt2_tokenizer, prompt, do_sample=True)
print(output_sample)
# Beklenen: Greedy'den daha yaratıcı ve çeşitli metin
# Her çalıştırmada farklı sonuç (seed olmadan)


### 6.4 Top-p (Nucleus) Sampling

**Top-p Sampling:** Kümülatif olasılık `p`'yi geçen tokenlardan oluşan "nucleus" (çekirdek) kümesinden örnekle.

Örnek `top_p=0.6`:
- Token olasılıkları sıralanır: [0.35, 0.20, 0.15, 0.10, ...]
- 0.35 + 0.20 = 0.55 < 0.6 → devam
- 0.35 + 0.20 + 0.15 = 0.70 > 0.6 → dur, bu 3 tokenden örnekle

- ✅ Çok düşük olasılıklı anlamsız tokenlar otomatik elenir
- ✅ Çeşitlilik korunur
- Greedy ve pure sampling arasında denge kurar


In [ ]:
torch.manual_seed(42)
output_topp = generate(gpt2, gpt2_tokenizer, prompt,
                       do_sample=True,
                       top_p=0.6)   # En olası %60'lık token kümesinden örnekle
print(output_topp)
# Beklenen: Pure sampling'den daha tutarlı, greedy'den daha yaratıcı


### 6.5 Prompt Engineering — Few-Shot ile Başkent Bulma

**Prompt Engineering:** Modeli fine-tune etmeden, sadece girdi metnini akıllıca tasarlayarak istenen davranışı elde etme.

**Few-Shot Learning:** Prompt'a birkaç örnek (few-shot) ekleyerek modele kalıbı gösterme.  
Model bu kalıbı tamamlamayı öğrenir — ek eğitim gerekmez.

Template: `"Capital city of France = Paris
Capital city of {country} ="`  
Modele bir örnek veriliyor → diğer ülkeler için de aynı kalıbı uygular.


In [ ]:
# Few-shot template: Modele kalıbı göster, tamamlamasını bekle
# {country}: f-string formatında doldurulacak değişken
DEFAULT_TEMPLATE = "Capital city of France = Paris\nCapital city of {country} ="

def get_capital_city(model, tokenizer, country, template=DEFAULT_TEMPLATE):
    """
    Few-shot prompt ile ülkenin başkentini tahmin ettirir.
    
    Çalışma mantığı:
    1. Template'e ülke adını ekle
    2. Model 10 yeni token üretsin
    3. Prompt'u çıkart, üretilen kısmın ilk satırını al
    """
    # Şablona ülke adını yerleştir
    prompt = template.format(country=country)

    # max_new_tokens=10: Kısa cevap yeterli (başkent 1-3 kelimedir)
    extended_text = generate(model, tokenizer, prompt, max_new_tokens=10)

    # Prompt'tan sonraki kısmı al
    answer = extended_text[len(prompt):]

    # İlk satırı al ve boşlukları temizle
    # Model "London\nCapital city of..." gibi devam edebilir
    return answer.strip().splitlines()[0].strip()


**Test: Başarılı Tahminler**

In [ ]:
result_uk = get_capital_city(gpt2, gpt2_tokenizer, "United Kingdom")
print(f"United Kingdom başkenti: {result_uk}")   # Beklenen: 'London'


In [ ]:
result_mx = get_capital_city(gpt2, gpt2_tokenizer, "Mexico")
print(f"Mexico başkenti: {result_mx}")   # Beklenen: 'Mexico City'


**Test: Esnek Giriş — Model'in Dayanıklılığı**

In [ ]:
# Model yazım yanlışlarını ve kısaltmaları da anlayabilir
results = [get_capital_city(gpt2, gpt2_tokenizer, country)
           for country in ("The UK", "Great Britain", "Big Britane")]
print(results)
# "Big Britane" (yanlış yazım) için bile mantıklı cevap verebilir


**Test: GPT-2'nin Bilgi Sınırları**

GPT-2 eğitim verisindeki yanlış bilgileri öğrenmiştir. Aşağıdaki ülkelerin başkentleri Canberra, Ottawa, Wellington ve Washington D.C.'dir — GPT-2 hata yapabilir.

In [ ]:
# GPT-2 bu ülkelerde yanılıyor — eğitim verisindeki yanlış bilgiler
results_wrong = [get_capital_city(gpt2, gpt2_tokenizer, country)
                 for country in ("Australia", "Canada", "New Zealand", "USA")]
print("GPT-2 tahminleri:", results_wrong)
print("Doğru cevaplar: Canberra, Ottawa, Wellington, Washington D.C.")


**Test: Belirsiz Girişler**

In [ ]:
# Model emin olmadığında ülke adını cevap olarak verir
results_unsure = [get_capital_city(gpt2, gpt2_tokenizer, country)
                  for country in ("Buthan", "Colombia", "Togo")]
print(results_unsure)


**Test: Anlamsız Girişler**

In [ ]:
# Ülke olmayan girişlerde model genellikle Paris'e döner
# Çünkü tek gördüğü örnek 'France = Paris' idi
results_nonsense = [get_capital_city(gpt2, gpt2_tokenizer, country)
                    for country in ("hey", "yo", "j")]
print(results_nonsense)
# Modelin "Paris bias'ı": Few-shot örnekteki tek şehir Paris


In [ ]:
# GPT-2'yi temizle — Mistral için yer aç
del model_id, gpt2
free_vram(device)


---
## 🔷 BÖLÜM 7: Mistral-7B — Büyük Dil Modeli

### Mistral-7B Nedir?

**Mistral-7B**, Mistral AI tarafından 2023'te yayınlanan 7 milyar parametreli açık kaynak modeldir.  
GPT-2 (117M) ile karşılaştırıldığında ~60× daha büyüktür.

**Teknik iyileştirmeler:**
- **Sliding Window Attention:** Uzun context'leri verimli işler
- **Grouped Query Attention (GQA):** Daha hızlı inference
- **SentencePiece tokenizer:** 32K vocabulary

> ⚠️ **Gereksinim:** Modeli indirmek için Hugging Face hesabı ve model kartını kabul etmek gerekir.


### 7.1 Hugging Face Access Token Yükleme

In [ ]:
# Hugging Face erişim token'ı — model indirmek için gerekli
# Token: https://huggingface.co/settings/tokens adresinden oluşturulur

if IS_COLAB:
    from google.colab import userdata
    # Colab'da 'Secrets' bölümünde saklanan token'ı güvenle al
    # Bu yöntem token'ı kod içine yazmaktan çok daha güvenlidir
    access_token = userdata.get('token-hf-read-mistral')
else:
    # Yerel ortamda: token'ı ayrı bir dosyada sakla (Git'e commit etme!)
    access_token = open("hf-read-mistral.secret").read().strip()
    # Alternatif: access_token = "hf_..."  # ⚠️ Güvenli değil — production'da kullanma


In [ ]:
from huggingface_hub import login

# Hugging Face Hub'a kimlik doğrula
# Token ~/.huggingface/token dosyasına kaydedilir
# Bir kez giriş yapıldıktan sonra aynı session'da tekrar giriş gerekmez
login(access_token)


### 7.2 Mistral-7B-v0.3 Modelini Yükleme

In [ ]:
# Mistral-7B base model (instruction-tuning olmayan ham model)
model_id = "mistralai/Mistral-7B-v0.3"

mistral7b_tokenizer = AutoTokenizer.from_pretrained(model_id)

# device_map='auto': Modeli uygun device'lara dağıt
# Eğer tek GPU varsa oraya, yoksa CPU'ya yükler
# dtype='auto': float16 ile yükle → ~14GB disk, ~14GB VRAM gerektirir
mistral7b = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", dtype="auto")


### 7.3 Mistral-7B ile Metin Üretimi

GPT-2 ile aynı few-shot başkent testini Mistral-7B ile deneyelim.

In [ ]:
torch.manual_seed(42)
output = generate(mistral7b, mistral7b_tokenizer, prompt, do_sample=True, top_p=0.6)
print(output)
# Mistral-7B çok daha bilgili ve tutarlı yanıtlar verir


**Mistral-7B Başkent Testi — Doğru Cevaplar**

In [ ]:
# Mistral-7B neredeyse tüm başkentleri doğru biliyor
results = [get_capital_city(mistral7b, mistral7b_tokenizer, country)
           for country in ("Australia", "Canada", "New Zealand", "USA")]
print("Mistral-7B tahminleri:", results)
print("Doğru cevaplar: Canberra, Ottawa, Wellington, Washington D.C.")


In [ ]:
# Belirsiz ülkeler — Mistral daha iyi tahmin eder
results_unsure = [get_capital_city(mistral7b, mistral7b_tokenizer, country)
                  for country in ("Buthan", "Colombia", "Togo")]
print(results_unsure)


### 7.4 Basit Chat Denemesi

Base model talimat takip etmez — metni sadece tamamlar.

In [ ]:
# Base model talimat değil, metin tamamlama yapar
# "List some places..." sorusunu cevaplamaz, konuşmayı devam ettirir
prompt = "List some places I should visit in Paris."
output = generate(mistral7b, mistral7b_tokenizer, prompt)
print(output)
# Beklenen: Model soruyu cevaplamaz, farklı bir konuşmayı devam ettirir


---
## 🔷 BÖLÜM 8: BobTheChatbot — Chatbot İnşa Etmek

### Base Model'den Chatbot'a

Base model metin tamamlayıcıdır. Chatbot davranışı elde etmek için:
1. Modele bir **sistem promptu (introduction)** vererek kim olduğunu söyle
2. Konuşmayı `Me: ... Bob: ...` formatında birleştir
3. Model bu kalıbı devam ettirir → chatbot etkisi

### Adım 1: Sistem Promptu


In [ ]:
# Sistem promptu: Modele kim olduğunu söyleyen bağlam metni
# Bu metin her konuşmada context'in başına eklenir
# "Bob" kimliği verilince model Bob gibi davranmaya çalışır
bob_introduction = """
Bob is an amazing chatbot. It knows everything and it's incredibly helpful.
"""


### Adım 2: Prompt + Yanıt Formatı

In [ ]:
# full_prompt: Sistem promptu + kullanıcı sorusu + Bob etiketi
# Model 'Bob:' etiketinden sonra cevabı üretecek
full_prompt = f"{bob_introduction}Me: {prompt}\nBob:"
extended_text = generate(mistral7b, mistral7b_tokenizer, full_prompt,
                         max_new_tokens=100)

# Prompt'u çıkart, sadece model'in ürettiği kısmı al
answer = extended_text[len(full_prompt):].strip()
print(answer)
# Sorun: Model Bob'un cevabından sonra konuşmayı kendisi devam ettirebilir:
# "Me: Tell me more... Bob: ..."
# Çözüm: İlk '\nMe:' işaretine kadar kes


In [ ]:
# Sadece Bob'un ilk cevabını al — hallucinated dialogue'u kes
clean_answer = answer.split("\nMe: ")[0]
print(clean_answer)


### Adım 3: Tam Chatbot Sınıfı — BobTheChatbot

**Context yönetimi:** Çok turlu konuşma için tüm geçmiş biriktirilir ve her soruda modele verilir.

**Döngülü üretim:** Model 100 token üretir. Cevap bitmemişse 100 token daha üretilir.  
Çıkış koşulları:
1. `\nMe:` görülürse → Bob'un cevabı bitti
2. Model hiçbir şey üretmezse → donmuş
3. Cevap çok uzarsa → güvenlik limiti


In [ ]:
class BobTheChatbot:
    """
    Decoder-only transformer tabanlı basit chatbot.
    
    Çalışma mantığı:
    - Konuşma geçmişini 'Me: ... Bob: ...' formatında string'de biriktirir
    - Her yeni soruda tüm geçmişi bağlam olarak modele verir
    - Model bu bağlamı kullanarak tutarlı çok turlu konuşma yapabilir
    
    Parametreler:
    - model:             Dil modeli (Mistral-7B vb.)
    - tokenizer:         Modele ait tokenizer
    - introduction:      Chatbot'u tanımlayan sistem promptu
    - max_answer_length: Sonsuz döngüye karşı güvenlik limiti
    """
    def __init__(self, model, tokenizer, introduction=bob_introduction,
                 max_answer_length=10_000):
        self.model = model
        self.tokenizer = tokenizer
        self.context = introduction    # Konuşma geçmişi burada birikir
        self.max_answer_length = max_answer_length

    def chat(self, prompt):
        """
        Kullanıcıdan mesaj al, cevap üret, döndür.
        Context otomatik güncellenir → bir sonraki soru için hazır.
        """
        # Yeni kullanıcı sorusunu ve Bob etiketini context'e ekle
        self.context += "\nMe: " + prompt + "\nBob:"
        context = self.context
        start_index = len(context)   # Bob'un cevabının başladığı indeks

        while True:
            # 100 token üret
            extended = generate(self.model, self.tokenizer, context,
                                max_new_tokens=100)
            answer = extended[start_index:]   # Sadece yeni üretilen kısım

            # Çıkış koşulları:
            if ("\nMe: " in answer or   # Bob'un cevabı bitti, kullanıcı devam etmiş
                extended == context or   # Model hiçbir şey üretmedi (donduk)
                len(answer) >= self.max_answer_length):   # Güvenlik limiti
                break
            context = extended   # Daha fazla token için bağlamı genişlet

        # Eğer model konuşmayı devam ettirdiyse, sadece Bob'un cevabını al
        answer = answer.split("\nMe: ")[0]

        # Bob'un cevabını kalıcı context'e ekle (bir sonraki tur için)
        self.context += answer
        return answer.strip()


### 8.1 Chatbot Testleri

In [ ]:
# Çok turlu konuşma testi
bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)

# Tur 1: Paris'teki yerler
answer1 = bob.chat("List some places I should visit in Paris.")
print("Bob:", answer1)


In [ ]:
# Tur 2: Context sayesinde 'first place'in ne olduğunu biliyor
answer2 = bob.chat("Tell me more about the first place.")
print("Bob:", answer2)


In [ ]:
# Tur 3: Context switch — Roma hakkında
# Model önceki konuşmayı "hatırlayarak" bağlantı kuruyor
answer3 = bob.chat("And Rome?")
print("Bob:", answer3)


### 8.2 Base Model'in Sınırlamaları

Base model (instruction-tuning olmadan) çeşitli sorunlara eğilimlidir.

**Sorun 1: Tekrar Döngüsü**

In [ ]:
# Base model uzun listeler oluşturmada tekrar döngüsüne düşebilir
bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)
print(bob.chat("Tell me 5 jokes"))
# Modelin aynı şakayı tekrar edebileceğini gözlemle


**Sorun 2: Yetersiz Yardım**

In [ ]:
# Base model belirsiz veya eksik cevaplar verebilir
bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)
print(bob.chat("How can I make cookies?"))
# Adım adım tarif yerine genel bir cevap verebilir


**Sorun 3: Zararlı İçerik (Hizalanmamış Model)**

In [ ]:
# Base model zararlı sorulara yanıt verebilir — hizalama yapılmamış
bad_bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)
bad_bob.chat("I'd like to rob a bank. How should I prepare?")
# Base model bu soruyu reddetmez — SFT + DPO ile hizalama gereklidir


**Sonuç:** Base modeli daha konuşkan ve yardımsever hale getirmek için **fine-tuning** gerekir.  
Bu genellikle iki aşamada yapılır:  
1. **SFT (Supervised Fine-Tuning):** Modele talimat takibini öğret  
2. **DPO (Direct Preference Optimization):** İnsan tercihlerine göre hizala


---
## 🔷 BÖLÜM 9: Log Probability Hesaplama — DPO'nun Temeli

### Neden Log Probability?

Modelin bir cevaba ne kadar "güvendiğini" ölçmek için log probability kullanılır.  
Bu, DPO kaybı hesaplamak için temeldir.

**Senaryo:** "The capital of Argentina is Buenos Aires" vs "The capital of Argentina is Madrid"  
Model Buenos Aires için daha yüksek probability vermeli → DPO bunu öğretir.


In [ ]:
# İki farklı tamamlamayı yan yana koyarak olasılıklarını karşılaştır
prompt = "The capital of Argentina is "
full_input = [prompt + "Buenos Aires", prompt + "Madrid"]

# Mistral-7B'nin ayrı PAD token'ı yok → EOS token'ı padding olarak kullan
mistral7b_tokenizer.pad_token = mistral7b_tokenizer.eos_token

# Her iki cümleyi aynı anda tokenize et (batch işlemi)
# padding=True: Kısa olan cümleyi otomatik doldur (eşit uzunluk için)
encodings = mistral7b_tokenizer(full_input, return_tensors="pt", padding=True)
encodings = encodings.to(device)

# Forward pass: model tüm pozisyonlar için sonraki token tahminlerini üretir
with torch.no_grad():
    logits = mistral7b(**encodings).logits  # (2, seq_len, vocab_size)
    # logits[0]: "The capital of Argentina is Buenos Aires" için tahminler
    # logits[1]: "The capital of Argentina is Madrid" için tahminler
    # logits boyutu: [2, 8, 32768] — 2 cümle, 8 token, 32768 vocab


In [ ]:
# Token ID'lerini görüntüle — hangi tokenlar var?
print("Token IDs:")
print(encodings.input_ids)
# Token #1 = start-of-sequence (BOS)
# Token #2 = end-of-sequence & padding token (EOS = PAD)


### 9.1 Yöntem 1: torch.gather ile Log Probability

In [ ]:
# Otomatik gerilimli modelde her pozisyondaki token, bir sonraki tokeni tahmin eder
# "Bir sonraki token" = mevcut tokendan bir pozisyon ileri

# Bir sonraki token ID'leri: input'u 1 pozisyon kaydır
# encodings.input_ids[:, 1:]: BOS token'ı atla, kalan tokenlar hedef
next_token_ids = encodings.input_ids[:, 1:]  # (2, seq_len-1)

# Logits'i log softmax ile log-olasılığa dönüştür
# log_softmax: Numerik kararlılık için (softmax + log yerine)
# [:, :-1]: Son tokeni atla (onun için "sonraki token" yok)
log_probas = F.log_softmax(logits, dim=-1)[:, :-1]  # (2, seq_len-1, vocab_size)

# torch.gather: Her pozisyondaki doğru tokenın log olasılığını seç
# dim=2: vocab boyutu üzerinden indeksleme
# index=next_token_ids.unsqueeze(2): (2, seq_len-1) → (2, seq_len-1, 1)
next_token_log_probas = torch.gather(
    log_probas, dim=2, index=next_token_ids.unsqueeze(2))
# Sonuç shape: (2, seq_len-1, 1)


### 9.2 Yöntem 2: cross_entropy ile Log Probability (Daha Verimli)

In [ ]:
# cross_entropy = -log(softmax(logits)[target]) — matematiksel olarak eşdeğer
# reduction='none': Her token için ayrı kayıp değeri (toplama/ortalama alma)
# permute(0,2,1): CrossEntropyLoss (B,C,L) formatı gerektirir
next_token_log_probas = -F.cross_entropy(
    logits[:, :-1].permute(0, 2, 1),   # (2, vocab_size, seq_len-1)
    next_token_ids,                     # (2, seq_len-1)
    reduction="none"                    # Her token için ayrı log-prob
)
# Sonuç: (2, seq_len-1) — her cümle için her token'ın log olasılığı


### 9.3 Token Olasılıklarını İnceleme

In [ ]:
# "The capital of Argentina is Buenos Aires" için token olasılıkları
probs_buenos = [f"{p.item():.2%}" for p in torch.exp(next_token_log_probas[0])]
print("'Buenos Aires' token olasılıkları:")
print(probs_buenos)
# Örnek: ['3.27%', '0.02%', '51.95%', '0.40%', '33.98%', '11.38%', '99.61%']
# Her değer: o pozisyondaki tokenın önceki context verilen olasılığı
# Son değer yüksek (%99.61) → "Aires" "Buenos"tan sonra çok olası ✓


In [ ]:
# "The capital of Argentina is Madrid" için token olasılıkları
probs_madrid = [f"{p.item():.2%}" for p in torch.exp(next_token_log_probas[1])]
print("'Madrid' token olasılıkları:")
print(probs_madrid)
# Örnek: ['0.14%', '3.27%', '0.02%', '51.95%', '0.37%', '32.03%', '0.00%']
# Son değer çok düşük (%0.00) → Model Madrid'i yanlış başkent biliyor ✓


### 9.4 Top-10 Sonraki Token Analizi

In [ ]:
# "is" tokenından sonra modelin en olası tahminleri neler?
# Bu, modelin "The capital of Argentina is ___" için ne düşündüğünü gösterir
token_index = 5  # "is" tokenının pozisyonu
topk = torch.topk(log_probas[0, token_index], k=10)

print("'is' tokenından sonra en olası 10 token:")
for v, i in zip(topk.values, topk.indices):
    print(f"{torch.exp(v).item():06.2%}  '{mistral7b_tokenizer.decode([i])}'")
# Beklenen: 'Buenos', ' the', ' Argentina' gibi tokenların üst sırada olması


### 9.5 Cevap Log Probability'si

'Buenos Aires' için sadece cevap token'larının toplam log olasılığı.

In [ ]:
# Sadece cevap token'larının log olasılığı: Son 2 token = 'Buenos' + 'Aires'
answer_log_proba = next_token_log_probas[0, -2:].sum()

# Olasılığa çevir
proba = torch.exp(answer_log_proba).item()
print(f"'Buenos Aires' olasılığı: {proba:.4f} ({proba:.2%})")
# Örnek: 0.1138 ≈ %11.4
# Bu düşük görünebilir ama Madrid'den çok daha yüksek olduğunu göreceğiz


### 9.6 Tüm Sequence'ın Toplam Log Probability'si

In [ ]:
# Tüm cümlenin toplam log probability'si (padding hariç)

# attention_mask[:, :-1]: Son pozisyon hariç maske (1=gerçek, 0=padding)
padding_mask = encodings.attention_mask[:, :-1]

# Padding tokenlarına sıfır ağırlık ver
log_probas_sum = (next_token_log_probas * padding_mask).sum(dim=1)
print(f"Toplam log probability:")
print(f"  'Buenos Aires': {log_probas_sum[0].item():.4f}")
print(f"  'Madrid':       {log_probas_sum[1].item():.4f}")
# Beklenen: tensor([-21.25, -30.25])
# Buenos Aires: -21.25 > Madrid: -30.25 → Model Buenos Aires'i tercih ediyor ✓
# Büyük (negatife daha yakın) log-prob = daha yüksek olasılık


### 9.7 `sum_of_log_probas()` ve `dpo_loss()` Fonksiyonları

Bu iki fonksiyon DPO eğitiminin çekirdeğidir:

**`sum_of_log_probas()`:** Tüm pipeline'ı tek fonksiyona toplar.

**`dpo_loss()`:** Direct Preference Optimization kaybını hesaplar.

**DPO Formülü:**
```
L_DPO = -log σ(β × [(log p_θ(y_c|x) - log p_ref(y_c|x)) - (log p_θ(y_r|x) - log p_ref(y_r|x))])
```

- **y_c:** Tercih edilen (chosen) cevap
- **y_r:** Reddedilen (rejected) cevap
- **p_θ:** Eğitilen model
- **p_ref:** Dondurulmuş referans model (SFT sonrası model)
- **β:** KL-divergence katsayısı — modelin referanstan çok uzaklaşmasını engeller


In [ ]:
def sum_of_log_probas(model, tokenizer, full_inputs):
    """
    Verilen cümlelerin toplam log probability'sini hesaplar.
    
    Padding maskesi uygulanarak PAD token'ları hariç tutulur.
    
    Parametreler:
    - model:       Dil modeli
    - tokenizer:   Tokenizer
    - full_inputs: Prompt + cevap içeren tam string listesi
    
    Döndürür: Her cümle için toplam log probability (tensor)
    """
    encodings = tokenizer(
        full_inputs, return_tensors="pt", padding=True).to(model.device)
    logits = model(**encodings).logits

    # Cross-entropy ile log-probability hesapla
    next_token_log_probas = -F.cross_entropy(
        logits[:, :-1].permute(0, 2, 1),
        encodings.input_ids[:, 1:],
        reduction="none")

    # Padding maskesi uygula ve topla
    return (next_token_log_probas * encodings.attention_mask[:, :-1]).sum(dim=1)


def dpo_loss(model, ref_model, tokenizer, full_input_c, full_input_r, beta=0.1):
    """
    Direct Preference Optimization (DPO) kayıp fonksiyonu.
    
    Amacı: Modeli 'chosen' cevabına daha yüksek, 'rejected' cevabına
    daha düşük log-probability verecek şekilde eğitmek.
    Referans modelden çok uzaklaşmamak için KL cezası uygulanır.
    
    Parametreler:
    - model:        Güncellenen model (eğitilen)
    - ref_model:    Dondurulmuş referans model (SFT sonrası)
    - tokenizer:    Tokenizer
    - full_input_c: Chosen (tercih edilen) prompt + cevap listesi
    - full_input_r: Rejected (reddedilen) prompt + cevap listesi
    - beta:         KL cezası katsayısı (küçük beta = daha fazla özgürlük)
    """
    # Eğitilen modelden log probability'ler
    p_c = sum_of_log_probas(model, tokenizer, full_input_c)  # chosen
    p_r = sum_of_log_probas(model, tokenizer, full_input_r)  # rejected

    # Referans modelden log probability'ler (gradyan hesaplanmaz — frozen)
    with torch.no_grad():
        p_ref_c = sum_of_log_probas(ref_model, tokenizer, full_input_c)
        p_ref_r = sum_of_log_probas(ref_model, tokenizer, full_input_r)

    # DPO loss:
    # (p_c - p_ref_c): Eğitilen model chosen'ı ref'e göre ne kadar daha fazla seviyor?
    # (p_r - p_ref_r): Eğitilen model rejected'ı ref'e göre ne kadar daha fazla seviyor?
    # Fark: Chosen'ın avantajı rejected'a göre
    # logsigmoid + negatif: Maximum likelihood hedefi
    return -F.logsigmoid(beta * ((p_c - p_ref_c) - (p_r - p_ref_r))).mean()


### 9.8 Fonksiyonları Test Etme

In [ ]:
# sum_of_log_probas testi
with torch.no_grad():
    result = sum_of_log_probas(mistral7b, mistral7b_tokenizer, full_input)
print(f"Buenos Aires log prob: {result[0].item():.4f}")
print(f"Madrid log prob:       {result[1].item():.4f}")
# Buenos Aires daha yüksek (daha az negatif) olmalı


In [ ]:
# dpo_loss testi — aynı modeli hem model hem ref_model olarak kullan
# (sadece API'yi test etmek için; gerçekte farklı modeller olmalı)
with torch.no_grad():
    loss = dpo_loss(mistral7b, mistral7b,   # model = ref_model (test amaçlı)
                    mistral7b_tokenizer,
                    [full_input[0]],         # chosen: Buenos Aires
                    [full_input[1]])          # rejected: Madrid
print(f"DPO Loss: {loss.item():.4f}")
# Aynı model kullanıldığında log(σ(0)) = log(0.5) ≈ -0.693 beklenir


In [ ]:
# Mistral-7B ve ilgili değişkenleri temizle
del_vars(["access_token", "answer", "answer_log_proba", "bad_bob", "bob",
          "bob_introduction", "encodings", "extended_text", "full_input",
          "full_prompt", "i", "log_probas", "log_probas_sum", "logits",
          "mistral7b", "mistral7b_tokenizer", "model_id", "next_token_ids",
          "next_token_log_probas", "padding_mask", "prompt", "token_index",
          "topk", "v"])


---
## 🔷 BÖLÜM 10: TRL Kütüphanesi ile Fine-Tuning

### SFT + DPO Pipeline'ı

Base modeli kullanışlı bir asistana dönüştürmek için:

1. **SFT (Supervised Fine-Tuning):** Talimat takibini öğret — "Human: soru → Assistant: cevap" kalıbı
2. **DPO (Direct Preference Optimization):** İnsan tercihlerine hizala — zararlı içeriği reddet, yardımsever ol

### TRL (Transformer Reinforcement Learning)

Hugging Face'in RL ve tercih öğrenimi için özelleşmiş kütüphanesi.  
`SFTTrainer` ve `DPOTrainer` sınıfları, karmaşık eğitim döngülerini birkaç satıra indirir.


### 10.1 Alpaca Dataset — SFT için Talimat Veri Seti

In [ ]:
# Stanford Alpaca: GPT-3 tarafından üretilmiş 52K talimat-cevap çifti
# İnsan-yazılmış gibi görünen high-quality instruction following data
sft_dataset = load_dataset("tatsu-lab/alpaca", split="train")


In [ ]:
# Orijinal Alpaca formatını incele
print(sft_dataset[1]["text"])
# Orijinal format çok uzun bir prefix içeriyor:
# "Below is an instruction that describes a task..."
# "### Instruction:"
# "### Response:"
# Bunu daha temiz bir formata dönüştüreceğiz


### 10.2 Dataset Preprocessing

In [ ]:
def preprocess(example):
    """
    Alpaca formatını daha temiz 'Human/Assistant' formatına dönüştürür.
    
    Orijinal alanlar:
    - instruction: Görev açıklaması
    - input:       Opsiyonel bağlam (boş olabilir)
    - output:      Beklenen cevap
    
    Yeni format:
    'Human: {instruction}\n-> {input (varsa)}\nAssistant: {output}'
    
    Bu kalıbı öğrenen model, 'Human:' gördüğünde soru,
    'Assistant:' gördüğünde cevap üretmeyi öğrenir.
    """
    text = f"Human: {example['instruction']}\n"

    # Opsiyonel bağlam varsa ekle (örn. "Şu metni özetle: ...")
    if example['input'] != "":
        text += f"-> {example['input']}\n"

    text += f"\nAssistant: {example['output']}"
    return {"text": text}

sft_dataset = sft_dataset.map(preprocess)


In [ ]:
# Dönüştürülmüş formatı kontrol et
print(sft_dataset[1]["text"])
# Beklenen:
# Human: What are the three primary colors?
#
# Assistant: The three primary colors are red, blue, and yellow.


In [ ]:
# Hızlı demo için dataset'i küçült (tam eğitimde bu satırı kaldır)
# shuffle: Veriyi karıştır, select: İlk 1000 örneği al
sft_dataset = sft_dataset.shuffle(seed=42).select(range(1000))


### 10.3 TRL Kurulumu

In [ ]:
# Colab/Kaggle'da TRL önceden kurulu değil
if IS_COLAB or IS_KAGGLE:
    %pip install -qU trl


### 10.4 SFTTrainer ile GPT-2'yi Fine-Tune Etme

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_model_dir = "./my_gpt2_sft_alpaca"  # Fine-tuned model kayıt klasörü

# SFT Eğitim Konfigürasyonu
training_args = SFTConfig(
    output_dir=sft_model_dir,        # Checkpoint'lar buraya kaydedilir
    max_length=512,                  # Maksimum sequence uzunluğu
    per_device_train_batch_size=4,   # GPU başına 4 örnek
    num_train_epochs=1,              # 1 tam epoch (hız için)
    save_steps=50,                   # Her 50 adımda checkpoint kaydet
    logging_steps=10,                # Her 10 adımda log yaz
    learning_rate=5e-5,              # Öğrenme hızı
    report_to="none"                 # WandB/TensorBoard'a raporlama yapma
)

# SFTTrainer:
# - 'gpt2': Base model ID — otomatik indirilir
# - train_dataset: Preprocess edilmiş Alpaca veri seti
# - SFT, yalnızca cevap kısmının kaybını hesaplar (soru kısmı yoksayılır)
sft_trainer = SFTTrainer("gpt2", train_dataset=sft_dataset, args=training_args)

# Eğitimi başlat
sft_train_output = sft_trainer.train()

# Fine-tuned modeli kaydet (DPO için kullanılacak)
sft_trainer.model.save_pretrained(sft_model_dir)


In [ ]:
# SFT nesnelerini temizle
del sft_dataset, training_args, sft_trainer, sft_train_output
free_vram(device)


### 10.5 DPO — Anthropic hh-rlhf Dataset

DPO için insan tercihlerini içeren bir dataset gerekir.

In [ ]:
# Anthropic hh-rlhf (helpful and harmless - RL from human feedback) dataset
# Gerçek insan değerlendirmecilerinin tercihlerini içerir:
# - chosen: Faydalı ve zararsız cevap (istenen davranış)
# - rejected: Zararlı veya yetersiz cevap (istenmeyen davranış)
pref_dataset = load_dataset("Anthropic/hh-rlhf", split="train")


In [ ]:
# Dataset yapısını incele
print("Veri alanları:", pref_dataset[2].keys())
# dict_keys(['chosen', 'rejected'])


In [ ]:
# Tercih edilen (güvenli) cevabı incele
print("CHOSEN (tercih edilen):")
print(pref_dataset[2]["chosen"].strip())
# Human: [zararlı soru]
# Assistant: "I really couldn't say..." → Zararsız, kaçınan cevap ✓


In [ ]:
# Reddedilen (zararlı) cevabı incele
print("REJECTED (reddedilen):")
print(pref_dataset[2]["rejected"].strip())
# Human: [zararlı soru]
# Assistant: [Zararlı bilgi veriyor] ✗
# DPO bu tür cevapların olasılığını düşürmeyi öğrenir


In [ ]:
# Hızlı demo için dataset'i küçült
pref_dataset = pref_dataset.shuffle(seed=42).select(range(100))


### 10.6 DPOTrainer ile Fine-Tuning

In [ ]:
from trl import DPOConfig, DPOTrainer

dpo_model_dir = "./my_gpt2_sft_alpaca_dpo_hh_rlhf"

# DPO Eğitim Konfigürasyonu
training_args = DPOConfig(
    output_dir=dpo_model_dir,
    max_length=512,
    per_device_train_batch_size=4,
    num_train_epochs=1,
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-5,    # SFT'den daha küçük lr: zaten iyi başlangıç noktasındayız
    report_to="none"
)

# GPT-2'nin PAD token'ı yok → EOS kullan
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

# DPOTrainer:
# - sft_model_dir: SFT ile fine-tune edilmiş model (başlangıç noktası)
# - DPOTrainer bu modelin bir kopyasını ref_model olarak otomatik oluşturur
# - chosen/rejected çiftleri üzerinde DPO loss minimizasyonu yapar
dpo_trainer = DPOTrainer(
    sft_model_dir,
    args=training_args,
    train_dataset=pref_dataset,    # chosen/rejected çiftleri
    processing_class=gpt2_tokenizer
)

# DPO eğitimini başlat
# Eğitim sürecinde:
# - 'chosen' cevapların log probability'si artar
# - 'rejected' cevapların log probability'si azalır
# - Ref modelden KL divergence kontrolü
dpo_train_output = dpo_trainer.train()

# Final DPO modelini kaydet
dpo_trainer.model.save_pretrained(dpo_model_dir)


In [ ]:
del_vars(["dpo_model_dir", "dpo_train_output", "dpo_trainer", "gpt2_tokenizer",
          "pref_dataset", "sft_model_dir", "training_args"])


---
## 🔷 BÖLÜM 11: Off-the-Shelf Instruction-Tuned Modeller

### Kendi Modelini Fine-Tune Etmek Zorunda Değilsin

SFT + DPO pipeline'ını anlamak önemlidir ama pratikte büyük şirketler bu işi zaten yapmış modelleri sunmaktadır.

**Mistral-7B-Instruct-v0.3:** Base modelin aksine, bu model:
- SFT ile talimat takibini öğrenmiş
- RLHF/DPO ile zararlı içeriği reddetmeyi öğrenmiş
- `[INST]...[/INST]` formatıyla özel chat template kullanır

Instruct modelleri "Bob" rolünü oynamaya gerek kalmadan doğrudan talimat alır.


In [ ]:
# Mistral-7B-Instruct: SFT + DPO ile hizalanmış model
model_id = "mistralai/Mistral-7B-Instruct-v0.3"
mistral7bi_tokenizer = AutoTokenizer.from_pretrained(model_id)
mistral7bi = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", dtype="auto")


### 11.1 Instruct Model ile Şaka Testi

Base model şaka söylerken tekrar döngüsüne düşüyordu. Instruct model 5 farklı şaka söyleyebilir.

In [ ]:
# Instruct model BobTheChatbot wrapper'ı ile de çalışır
good_bob = BobTheChatbot(mistral7bi, mistral7bi_tokenizer)
print(good_bob.chat("Tell me 5 jokes"))
# Beklenen: 5 farklı ve mantıklı şaka — tekrar yok ✓


### 11.2 Kurabiye Tarifi Testi

Base model belirsiz cevap veriyordu. Instruct model adım adım tarif verir.

In [ ]:
good_bob = BobTheChatbot(mistral7bi, mistral7bi_tokenizer)
print(good_bob.chat("How can I make cookies?"))
# Beklenen: Malzemeler ve adım adım hazırlık talimatları ✓


### 11.3 Güvenlik Testi

Base model zararlı soruya cevap veriyordu. Instruct/Aligned model reddeder.

In [ ]:
good_bob = BobTheChatbot(mistral7bi, mistral7bi_tokenizer)
good_bob.chat("I'd like to rob a bank. How should I prepare?")
# Beklenen: Modelin bu soruyu nazikçe reddetmesi ✓
# DPO eğitimi zararlı soruları reddetmeyi öğretti


In [ ]:
# Instruct modeli temizle
del good_bob, mistral7bi, mistral7bi_tokenizer, model_id
free_vram(device)


---
## 🔷 EK MATERYAL: Sabit Positional Encoding (Sin/Cos)

### Orijinal Makale Yaklaşımı

Orijinal "Attention Is All You Need" makalesi, **öğrenilebilir değil sabit** pozisyonel kodlama kullandı.

**Formül:**
$$\mathrm{PE}(p,i) = \begin{cases} \sin(p / 10000^{i/d}) & i \text{ çift} \\ \cos(p / 10000^{(i-1)/d}) & i \text{ tek} \end{cases}$$

- **p:** Token'ın cümledeki pozisyonu (0, 1, 2, ...)
- **i:** Encoding vektörünün boyutu indeksi
- **d:** Toplam embedding boyutu

### Neden Sin/Cos?

1. **Sınırlı:** -1 ile +1 arasında → büyük değerlere sahip olmaz
2. **Göreceli pozisyon:** `cos(p + Δ)` ve `sin(p + Δ)`, `cos(p)` ve `sin(p)` lineer kombinasyonu olarak ifade edilebilir → model göreceli pozisyonları öğrenebilir
3. **Çok frekans:** Farklı ölçeklerdeki örüntüler için zengin temsil

### Öğrenilebilir vs. Sabit

| Özellik | Sabit (Sin/Cos) | Öğrenilebilir |
|---------|-----------------|----------------|
| Parametre | ❌ Yok | ✅ var |
| Küçük veri seti | ✅ İyi | ❌ Ezber riski |
| Büyük veri seti | ❌ Sınırlı | ✅ Daha iyi |
| Günümüz kullanımı | RoPE/ALiBi | Tercih edilir |


In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sabit (öğrenilemez) Positional Encoding — orijinal Transformer makalesi.
    
    Sin ve cosine fonksiyonlarını farklı frekanslarda kullanarak
    her pozisyon için benzersiz bir vektör üretir.
    
    Öğrenilebilir PositionalEmbedding'den farkı:
    - nn.Parameter kullanmaz → optimizer güncellemiyor
    - register_buffer kullanır → model.state_dict()'e dahil olur ama eğitilmez
    """
    def __init__(self, max_length, embed_dim, dropout=0.1):
        super().__init__()

        # p: Pozisyon indeksleri (0, 1, 2, ..., max_length-1) — sütun vektörü
        p = torch.arange(max_length).unsqueeze(1)  # (max_length, 1)

        # i: Çift boyut indeksleri (0, 2, 4, ..., embed_dim-2)
        i = torch.arange(0, embed_dim, 2)  # (embed_dim // 2,)

        # Açı: p / 10000^(i/embed_dim)
        # Yüksek i değerlerinde frekans çok düşük (yavaş salınım)
        # Düşük i değerlerinde frekans yüksek (hızlı salınım)
        angle = p / 10_000 ** (i / embed_dim)  # (max_length, embed_dim // 2)

        # Encoding matrisini doldur
        pos_encodings = torch.empty(max_length, embed_dim)
        pos_encodings[:, ::2]  = angle.sin()   # Çift indeksler: sin
        pos_encodings[:, 1::2] = angle.cos()   # Tek indeksler: cos

        # register_buffer: Bu tensor eğitilmez ama model state'ine dahil
        # .to(device) çağrısında otomatik GPU'ya taşınır
        self.register_buffer("pos_encodings", pos_encodings)

        self.dropout = nn.Dropout(dropout)

    def forward(self, X):
        """
        X: (batch_size, seq_len, embed_dim)
        X'e pozisyonel encoding ekle ve dropout uygula.
        """
        return self.dropout(X + self.pos_encodings[:X.size(1)])


### Ek: Kullanım Örneği

In [ ]:
max_length = 500
embed_dim = 512
pos_encoding = PositionalEncoding(max_length, embed_dim)

# Test: 256 cümle, 500 token, 512 boyutlu
embeddings = torch.randn(256, 500, 512)
embeddings_with_pos = pos_encoding(embeddings)

print(f"Giriş boyutu:  {embeddings.shape}")
print(f"Çıkış boyutu: {embeddings_with_pos.shape}")
# İkisi de (256, 500, 512) — şekil değişmez


### Ek: Positional Encoding Matrisini Görselleştirme

Bu grafik, sin/cos encoding matrisinin nasıl göründüğünü ve iki tokenın  
`i=100` ve `i=101` boyutlarındaki encoding değerlerini karşılaştırır.


In [ ]:
figure_max_length = 201
figure_enc_size = 512
pos_enc = PositionalEncoding(figure_max_length, figure_enc_size)
P = pos_enc.pos_encodings.numpy()   # NumPy matrisine çevir

# Görselleştirme parametreleri
i1, i2, crop_i = 100, 101, 150    # İncelenecek encoding boyutları
p1, p2, p3 = 22, 60, 35           # İncelenecek pozisyonlar

fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(9, 5))

# Üst grafik: Encoding matrisinin ısı haritası
# x ekseni: Token pozisyonu (p)
# y ekseni: Encoding boyutu (i)
img = ax1.imshow(P.T[:crop_i], cmap="gray", interpolation="bilinear", aspect="auto")
cheat = 2  # Mavi çizgiyi gizlememek için kırmızı çizgi biraz yukarı
ax1.set_title("Sine/cosine positional encoding matrix (transposed)")
ax1.hlines(i2+cheat, 0, figure_max_length - 1, color="r", linewidth=3)  # i=101 (kırmızı)
ax1.hlines(i1, 0, figure_max_length - 1, color="b", linewidth=3)         # i=100 (mavi)
ax1.plot([p1, p1], [0, crop_i], "k--")   # p=22 dikey çizgi
ax1.plot([p2, p2], [0, crop_i], "k--", alpha=0.5)  # p=60 dikey çizgi
ax1.plot([p1, p2], [i2+cheat, i2+cheat], "ro")  # i=101'deki noktalar
ax1.plot([p1, p2], [i1, i1], "bo")               # i=100'deki noktalar
ax1.axis([0, figure_max_length - 1, 0, crop_i])
ax1.set_ylabel("Dim. $i$", rotation=0, fontsize=12)

# Alt grafik: i=100 (sin) ve i=101 (cos) encoding'lerinin dalga formu
ax2.set_title("Zoom on $i = 100$ and $i = 101$")
ax2.plot([p1, p1], [-1, 1], "k--", label="$p = {}$".format(p1))
ax2.plot([p2, p2], [-1, 1], "k--", label="$p = {}$".format(p2), alpha=0.5)
ax2.plot(p3, P[p3, i1], "bx", label="$p = {}$".format(p3))
ax2.plot(P[:,i1], "b-", label="$i = {}$".format(i1))   # sin dalgası
ax2.plot(P[:,i2], "r-", label="$i = {}$".format(i2))   # cos dalgası
ax2.plot([p1, p2], [P[p1, i1], P[p2, i1]], "bo")
ax2.plot([p1, p2], [P[p1, i2], P[p2, i2]], "ro")
ax2.set_xlabel("Token position $p$", fontsize=12)
ax2.legend(loc="center right", fontsize=14, framealpha=0.95)
ax2.set_ylabel("$PE(p,i)$", rotation=0, fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.hlines(0, 0, figure_max_length - 1, color="k", linewidth=1, alpha=0.3)
ax2.axis([0, figure_max_length - 1, -1, 1])
plt.tight_layout()
plt.show()

# Grafik yorumu:
# Üst: Her pozisyon için benzersiz bir pattern → model pozisyonu ayırt edebilir
# Alt: i=100 ve i=101 boyutları p=22 ve p=60'ta farklı değerler alıyor
#      Δ=38 token mesafedeki iki pozisyon sabit fark üretiyor → göreceli pozisyon öğrenilebilir


In [ ]:
# Bellekte kalan tüm değişkenleri temizle
del_vars(["max_length", "embed_dim", "pos_encoding", "embeddings",
          "embeddings_with_pos", "figure_max_length", "figure_enc_size",
          "pos_enc", "P", "i1", "i2", "crop_i", "p1", "p2", "p3", "fig",
          "ax1", "ax2", "img", "cheat"])


---
## 📋 BÖLÜM ÖZETİ

| Konu | Öğrendiklerimiz |
|------|-----------------|
| **PositionalEmbedding** | Token sırası bilgisi için öğrenilebilir konum vektörleri |
| **MultiheadAttention** | Q/K/V projeksiyonları, scaled dot-product, h başlık |
| **Attention Masking** | Causal mask (gelecek engeli) + padding mask |
| **EncoderLayer** | Self-attention + FFN + 2× Residual + LayerNorm |
| **DecoderLayer** | Masked self-attention + Cross-attention + FFN |
| **TransformerEncoder/Decoder** | deepcopy ile çok katmanlı yığma |
| **NmtTransformer** | Tam İng→İsp çeviri modeli, causal mask oluşturma |
| **BERT MLM** | BertConfig, DataCollatorForLanguageModeling, Trainer API |
| **SBERT** | Cümle embedding'leri, cosine similarity |
| **GPT-2 / Mistral-7B** | Causal LM, generate(), sampling stratejileri |
| **Few-shot Prompting** | Template ile sıfır fine-tuning |
| **BobTheChatbot** | Context yönetimi, döngülü üretim, çok turlu diyalog |
| **Log Probability** | torch.gather vs cross_entropy, padding mask |
| **DPO Loss** | sum_of_log_probas, chosen vs rejected, beta katsayısı |
| **SFTTrainer** | Alpaca dataset preprocessing, talimat kalıbı öğretimi |
| **DPOTrainer** | hh-rlhf dataset, tercih öğrenimi |
| **Instruct Model** | Off-the-shelf hizalanmış model kullanımı |
| **Fixed Positional Enc.** | Sin/cos formülü, register_buffer, görselleştirme |

---

## 🔗 Kaynaklar

- Vaswani et al., ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762) (2017)
- Devlin et al., ["BERT: Pre-Training of Deep Bidirectional Transformers"](https://arxiv.org/abs/1810.04805) (2018)
- Reimers & Gurevych, ["Sentence-BERT"](https://arxiv.org/abs/1908.10084) (2019)
- Radford et al., ["Improving Language Understanding by Generative Pre-Training"](https://openai.com/research/language-unsupervised) (2018)
- Rafailov et al., ["Direct Preference Optimization"](https://arxiv.org/abs/2305.18290) (2023)
- [Hugging Face Transformers Docs](https://huggingface.co/docs/transformers)
- [TRL Library](https://huggingface.co/docs/trl)
- [Sentence Transformers](https://www.sbert.net/)
